In [ ]:
import os
os.chdir("/Users/nirjharbhattacharyya/Downloads/Nirjhar_papers_Ferdig_lab/MKN_BD_popgen")

Association of KIC1 using PC50

In [ ]:
import re
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

from scipy.stats import ttest_ind, mannwhitneyu, pearsonr, spearmanr, ranksums
from sklearn.preprocessing import StandardScaler

# ============================================================
# FILE
# ============================================================
geno = pd.read_csv("KIC_1_vcf_subset_BD - Sheet1.csv")

# ============================================================
# PC50 VALUES
# ============================================================
pc50 = pd.DataFrame({
    "Sample": [
        "I-001-clone", "I-002", "I-003-clone", "I-004", "I-005",
        "I-008", "I-010", "I-011", "I-013", "I-016",
        "I-017", "I-018", "I-019", "I-020", "I-021",
        "I-025", "I-026", "I-027", "I-029", "I-030",
        "I-031", "I-033", "I-035", "I-036", "I-039",
        "I-041"
    ],
    "PC50": [
        0.015, 3.56, 5.967, 1.6, 3.82,
        2.71, 2.01, 5.35, 7.98, 10.25,
        4.51, 3.42, 2.02, 7.66, 2.83,
        3.39, 3.25, 3.23, 7.68, 8.19,
        3.41, 8.48, 1.25, 9.42, 13.14,
        4.82
    ]
})

pc50["Sample"] = pc50["Sample"].astype(str).str.strip()
pc50["PC50"] = pd.to_numeric(pc50["PC50"], errors="coerce")
pc50 = pc50.dropna()

# ============================================================
# MATCH SAMPLES
# ============================================================
if "FORMAT" not in geno.columns:
    raise ValueError("FORMAT column not found in genotype file.")

format_idx = list(geno.columns).index("FORMAT")
samples = list(geno.columns[format_idx + 1:])
samples = [str(s).strip() for s in samples]

common = sorted(set(samples).intersection(set(pc50["Sample"])))

print("Samples in VCF:", len(samples))
print("Samples with PC50:", len(pc50))
print("Matched samples:", len(common))
print(common)

if len(common) == 0:
    raise ValueError("No matching sample names between genotype file and PC50 table.")

pc50_data = pc50.set_index("Sample").loc[common]

# ============================================================
# SNP IDs
# ============================================================
chrom_col = "#CHROM" if "#CHROM" in geno.columns else "CHROM"

geno["POS"] = pd.to_numeric(geno["POS"], errors="coerce")
geno = geno.dropna(subset=["POS"]).copy()
geno["POS"] = geno["POS"].astype(int)

geno["SNP_ID"] = geno.apply(
    lambda r: f"{r[chrom_col]}:{int(r['POS'])}:{r['REF']}>{r['ALT']}",
    axis=1
)

# ============================================================
# ALT MATRIX
# ============================================================
def alt_present(x):
    if pd.isna(x):
        return 0.0

    gt = str(x).split(":")[0]

    if gt in ["./.", ".|.", "."]:
        return 0.0

    alleles = re.split(r"[\/|]", gt)
    return 1.0 if "1" in alleles else 0.0

A = pd.DataFrame(
    {s: geno[s].map(alt_present).values for s in common},
    index=geno["SNP_ID"]
).T

# ============================================================
# PARSE EFFECT FROM ANN
# ============================================================
def get_effect(info):
    if pd.isna(info):
        return None

    for field in str(info).split(";"):
        if field.startswith("ANN="):
            ann = field[4:].split(",")[0]
            parts = ann.split("|")
            if len(parts) > 1:
                return parts[1]

    return None

geno["EFFECT"] = geno["INFO"].apply(get_effect)

# ============================================================
# KEEP ALL RELEVANT SNPs:
# missense + stop_gained + intergenic
# ============================================================
keep = geno["EFFECT"].str.contains(
    "missense|stop_gained|intergenic",
    na=False
)

geno_filt = geno[keep].copy()

print("\nNumber of retained SNPs:", geno_filt.shape[0])
print("\nRetained SNPs:")
for s in geno_filt["SNP_ID"].tolist():
    print(" ", s)

A_filt = A[geno_filt["SNP_ID"]].copy()

# ============================================================
# SKAT-LIKE SQUARED MAF-WEIGHTED GENE SCORE
# Original score: sum(w_j * G_ij)
# New SKAT-like score: sum((w_j * G_ij)^2)
# Since G is binary 0/1, this becomes sum(w_j^2 for present ALT SNPs)
# ============================================================
maf = A_filt.mean(axis=0)

weights = 1 / np.sqrt(maf * (1 - maf))
weights = weights.replace([np.inf, -np.inf], 0).fillna(0)

weighted_alt = A_filt * weights
gene_score = (weighted_alt ** 2).sum(axis=1)

df = pd.DataFrame({
    "Sample": gene_score.index,
    "Score": gene_score.values,
    "PC50": pc50_data.loc[gene_score.index, "PC50"].values
})

df["Group"] = np.where(df["PC50"] >= 5, "Delayed_PC50", "Low_PC50")
df["Binary"] = (df["PC50"] >= 5).astype(int)

df = df.dropna(subset=["PC50", "Score"]).copy()

print("\nAnalysis dataframe:")
print(df.to_string(index=False))

if df["Score"].nunique() <= 1:
    raise ValueError("Score has no variation across samples.")

# ============================================================
# CONTINUOUS ASSOCIATION: PC50 ~ SCORE
# ============================================================
X = sm.add_constant(df["Score"])
fit = sm.OLS(df["PC50"], X).fit()

pearson_r, pearson_p = pearsonr(df["Score"], df["PC50"])
spearman_r, spearman_p = spearmanr(df["Score"], df["PC50"])

# ============================================================
# GROUP TESTS
# ============================================================
low = df[df["Group"] == "Low_PC50"]["Score"]
delayed = df[df["Group"] == "Delayed_PC50"]["Score"]

welch_p = ttest_ind(low, delayed, equal_var=False).pvalue
mw_p = mannwhitneyu(low, delayed, alternative="two-sided").pvalue
wilcox_p = ranksums(low, delayed).pvalue

# ============================================================
# LOGISTIC REGRESSION
# ============================================================
try:
    logit = sm.Logit(df["Binary"], sm.add_constant(df["Score"])).fit(disp=0)
    logit_beta = logit.params.iloc[1]
    logit_p = logit.pvalues.iloc[1]
    odds_ratio = np.exp(logit_beta)
except Exception as e:
    logit_beta = np.nan
    logit_p = np.nan
    odds_ratio = np.nan
    print("\nLogistic regression failed:", e)

# ============================================================
# PRINT RESULTS
# ============================================================
print("\n" + "=" * 70)
print("SKAT-LIKE SQUARED MAF-WEIGHTED KIC1 SCORE USING PC50")
print("=" * 70)

print("\nWeights:")
print(weights.to_string())

print("\nSquared weights:")
print((weights ** 2).to_string())

print("\n--- Continuous association: PC50 ~ Score ---")
print(f"OLS beta = {fit.params.iloc[1]:.12f}")
print(f"OLS p = {fit.pvalues.iloc[1]:.12g}")
print(f"R^2 = {fit.rsquared:.12f}")
print(f"Pearson r = {pearson_r:.12f}, p = {pearson_p:.12g}")
print(f"Spearman rho = {spearman_r:.12f}, p = {spearman_p:.12g}")

print("\n--- Group tests: PC50 < 5 h vs PC50 >= 5 h ---")
print(f"Welch t-test p = {welch_p:.12g}")
print(f"Mann-Whitney p = {mw_p:.12g}")
print(f"Wilcoxon rank-sum p = {wilcox_p:.12g}")

print("\n--- Logistic regression: Delayed PC50 ~ Score ---")
print(f"Logistic beta = {logit_beta}")
print(f"Logistic p = {logit_p}")
print(f"Odds ratio = {odds_ratio}")

print("\n--- Group means ---")
print(f"Low PC50 mean score = {low.mean():.12f}")
print(f"Delayed PC50 mean score = {delayed.mean():.12f}")

print("\n--- Sample-level scores ---")
print(df.to_string(index=False))

# ============================================================
# PLOT 1: SCORE VS PC50 WITH SAMPLE LABELS
# ============================================================
plt.figure(figsize=(8, 6))

plt.scatter(df["Score"], df["PC50"], s=70)

xline = np.linspace(df["Score"].min(), df["Score"].max(), 100)
yline = fit.params.iloc[0] + fit.params.iloc[1] * xline
plt.plot(xline, yline)

plt.axhline(5, linestyle="--", linewidth=1)

for _, row in df.iterrows():
    plt.text(
        row["Score"] + 0.03,
        row["PC50"],
        row["Sample"],
        fontsize=7,
        alpha=0.8
    )

plt.xlabel("SKAT-like squared MAF-weighted KIC1 score")
plt.ylabel("PC50 (h)")
plt.title(f"KIC1 score vs PC50\nOLS p={fit.pvalues.iloc[1]:.2e}, R²={fit.rsquared:.2f}")

plt.tight_layout()
plt.show()

# ============================================================
# PLOT 2: VIOLIN + BOX + POINTS + SAMPLE LABELS
# ============================================================
plt.figure(figsize=(9, 6))

df_plot = df.copy()

plot_groups = [
    df_plot[df_plot["Group"] == "Low_PC50"],
    df_plot[df_plot["Group"] == "Delayed_PC50"]
]

pos = [1, 2]

plt.violinplot(
    [d["Score"] for d in plot_groups],
    positions=pos,
    showextrema=False
)

bp = plt.boxplot(
    [d["Score"] for d in plot_groups],
    positions=pos,
    widths=0.25,
    patch_artist=True,
    showfliers=False
)

for b in bp["boxes"]:
    b.set_facecolor("none")

rng = np.random.default_rng(0)

for i, d in enumerate(plot_groups):
    x_center = pos[i]
    xj = np.full(len(d), x_center, dtype=float) + rng.uniform(-0.09, 0.09, len(d))
    yj = d["Score"].values
    labels = d["Sample"].values

    plt.scatter(xj, yj, s=55)

    for x, y, label in zip(xj, yj, labels):
        plt.text(
            x + 0.025,
            y,
            label,
            fontsize=7,
            alpha=0.85
        )

plt.xticks(pos, ["PC50 < 5 h", "PC50 ≥ 5 h"])
plt.ylabel("SKAT-like squared MAF-weighted KIC1 score")
plt.title(f"KIC1 score by PC50 group\nWelch p={welch_p:.2e} | MW p={mw_p:.2e}")

plt.tight_layout()
plt.show()

# ============================================================
# PLOT 3: RANKED BAR PLOT WITH SAMPLE NAMES
# ============================================================
plt.figure(figsize=(10, 5))

df_sorted = df.sort_values("Score").reset_index(drop=True)
colors = ["grey" if g == "Low_PC50" else "black" for g in df_sorted["Group"]]

plt.bar(range(len(df_sorted)), df_sorted["Score"], color=colors)

plt.xticks(range(len(df_sorted)), df_sorted["Sample"], rotation=90)
plt.ylabel("SKAT-like squared MAF-weighted KIC1 score")
plt.xlabel("Samples")
plt.title("Ranked KIC1 scores: Low PC50 = grey, Delayed PC50 = black")

plt.tight_layout()
plt.show()

# ============================================================
# PERMUTATION TEST
# ============================================================
obs_diff = delayed.mean() - low.mean()

perm_diffs = []
rng = np.random.default_rng(1)

for _ in range(10000):
    shuffled = rng.permutation(df["Score"].values)
    perm_low = shuffled[:len(low)]
    perm_delayed = shuffled[len(low):]
    perm_diffs.append(perm_delayed.mean() - perm_low.mean())

p_perm = np.mean(np.abs(perm_diffs) >= abs(obs_diff))

print("\nPermutation p =", p_perm)

# ============================================================
# LOG-TRANSFORMED SCORE TEST
# ============================================================
df["Score_log"] = np.log1p(df["Score"])

low_log = df[df["Group"] == "Low_PC50"]["Score_log"]
delayed_log = df[df["Group"] == "Delayed_PC50"]["Score_log"]

print("\n--- Log-score group tests ---")
print("Welch p =", ttest_ind(low_log, delayed_log, equal_var=False).pvalue)
print("Mann-Whitney p =", mannwhitneyu(low_log, delayed_log, alternative="two-sided").pvalue)

# ============================================================
# STANDARDIZED LOGISTIC REGRESSION
# ============================================================
scaler = StandardScaler()
df["Score_z"] = scaler.fit_transform(df[["Score"]])

try:
    X_z = sm.add_constant(df["Score_z"])
    y_bin = df["Binary"]

    logit_z = sm.Logit(y_bin, X_z).fit(disp=0)

    print("\n" + "=" * 60)
    print("STANDARDIZED LOGISTIC REGRESSION")
    print("=" * 60)
    print("Coefficient =", logit_z.params["Score_z"])
    print("p-value =", logit_z.pvalues["Score_z"])
    print("Odds ratio =", np.exp(logit_z.params["Score_z"]))

    conf = logit_z.conf_int().loc["Score_z"]
    conf_odds = np.exp(conf)
    print("95% CI odds ratio =", tuple(conf_odds))

    print("\nFull summary:")
    print(logit_z.summary())

except Exception as e:
    print("\nStandardized logistic regression failed:", e)

# ============================================================
# PLOTTING BLOCK WITHOUT SAMPLE NAME LABELS
# ============================================================

# ------------------------------------------------------------
# PLOT 1: SCORE VS PC50, NO LABELS
# ------------------------------------------------------------
plt.figure(figsize=(8, 6))

plt.scatter(df["Score"], df["PC50"], s=70)

xline = np.linspace(df["Score"].min(), df["Score"].max(), 100)
yline = fit.params.iloc[0] + fit.params.iloc[1] * xline
plt.plot(xline, yline)

plt.axhline(5, linestyle="--", linewidth=1)

plt.xlabel("SKAT-like squared MAF-weighted KIC1 score")
plt.ylabel("PC50 (h)")
plt.title(f"KIC1 score vs PC50\nOLS p={fit.pvalues.iloc[1]:.2e}, R²={fit.rsquared:.2f}")

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# PLOT 2: VIOLIN + BOX + POINTS, NO LABELS
# ------------------------------------------------------------
plt.figure(figsize=(9, 6))

data = [low.values, delayed.values]
pos = [1, 2]

plt.violinplot(data, positions=pos, showextrema=False)

bp = plt.boxplot(
    data,
    positions=pos,
    widths=0.25,
    patch_artist=True,
    showfliers=False
)

for b in bp["boxes"]:
    b.set_facecolor("none")

rng = np.random.default_rng(0)

for i, arr in enumerate(data):
    xj = np.full(len(arr), pos[i], dtype=float) + rng.uniform(-0.09, 0.09, len(arr))
    plt.scatter(xj, arr, s=55)

plt.xticks(pos, ["PC50 < 5 h", "PC50 ≥ 5 h"])
plt.ylabel("SKAT-like squared MAF-weighted KIC1 score")
plt.title(f"KIC1 score by PC50 group\nWelch p={welch_p:.2e} | MW p={mw_p:.2e}")

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# PLOT 3: RANKED BAR PLOT, NO SAMPLE NAMES ON BARS
# ------------------------------------------------------------
plt.figure(figsize=(10, 5))

df_sorted = df.sort_values("Score").reset_index(drop=True)
colors = ["grey" if g == "Low_PC50" else "black" for g in df_sorted["Group"]]

plt.bar(range(len(df_sorted)), df_sorted["Score"], color=colors)

plt.ylabel("SKAT-like squared MAF-weighted KIC1 score")
plt.xlabel("Samples ranked")
plt.title("Ranked KIC1 scores: Low PC50 = grey, Delayed PC50 = black")

plt.tight_layout()
plt.show()


print("\nSNPs used in the KIC1 score:")
for i, row in geno_filt.iterrows():
    print(
        row["SNP_ID"],
        "| EFFECT:",
        row["EFFECT"],
        "| REF:",
        row["REF"],
        "| ALT:",
        row["ALT"]
    )

In [ ]:
import re
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

from scipy.stats import ttest_ind, mannwhitneyu, pearsonr, spearmanr, ranksums
from sklearn.preprocessing import StandardScaler

# ============================================================
# FILE
# ============================================================
geno = pd.read_csv("KIC_1_vcf_subset_BD - Sheet1.csv")

# ============================================================
# PC50 VALUES
# ============================================================
pc50 = pd.DataFrame({
    "Sample": [
        "I-001-clone", "I-002", "I-003-clone", "I-004", "I-005",
        "I-008", "I-010", "I-011", "I-013", "I-016",
        "I-017", "I-018", "I-019", "I-020", "I-021",
        "I-025", "I-026", "I-027", "I-029", "I-030",
        "I-031", "I-033", "I-035", "I-036", "I-039",
        "I-041"
    ],
    "PC50": [
        0.015, 3.56, 5.967, 1.6, 3.82,
        2.71, 2.01, 5.35, 7.98, 10.25,
        4.51, 3.42, 2.02, 7.66, 2.83,
        3.39, 3.25, 3.23, 7.68, 8.19,
        3.41, 8.48, 1.25, 9.42, 13.14,
        4.82
    ]
})

pc50["Sample"] = pc50["Sample"].astype(str).str.strip()
pc50["PC50"] = pd.to_numeric(pc50["PC50"], errors="coerce")
pc50 = pc50.dropna()

# ============================================================
# MATCH SAMPLES
# ============================================================
if "FORMAT" not in geno.columns:
    raise ValueError("FORMAT column not found in genotype file.")

format_idx = list(geno.columns).index("FORMAT")
samples = list(geno.columns[format_idx + 1:])
samples = [str(s).strip() for s in samples]

common = sorted(set(samples).intersection(set(pc50["Sample"])))

print("Samples in VCF:", len(samples))
print("Samples with PC50:", len(pc50))
print("Matched samples:", len(common))
print(common)

if len(common) == 0:
    raise ValueError("No matching sample names between genotype file and PC50 table.")

pc50_data = pc50.set_index("Sample").loc[common]

# ============================================================
# SNP IDs
# ============================================================
chrom_col = "#CHROM" if "#CHROM" in geno.columns else "CHROM"

geno["POS"] = pd.to_numeric(geno["POS"], errors="coerce")
geno = geno.dropna(subset=["POS"]).copy()
geno["POS"] = geno["POS"].astype(int)

geno["SNP_ID"] = geno.apply(
    lambda r: f"{r[chrom_col]}:{int(r['POS'])}:{r['REF']}>{r['ALT']}",
    axis=1
)

# ============================================================
# ALT MATRIX
# ============================================================
def alt_present(x):
    if pd.isna(x):
        return 0.0

    gt = str(x).split(":")[0]

    if gt in ["./.", ".|.", "."]:
        return 0.0

    alleles = re.split(r"[\/|]", gt)
    return 1.0 if "1" in alleles else 0.0

A = pd.DataFrame(
    {s: geno[s].map(alt_present).values for s in common},
    index=geno["SNP_ID"]
).T

# ============================================================
# PARSE EFFECT FROM ANN
# ============================================================
def get_effect(info):
    if pd.isna(info):
        return None

    for field in str(info).split(";"):
        if field.startswith("ANN="):
            ann = field[4:].split(",")[0]
            parts = ann.split("|")
            if len(parts) > 1:
                return parts[1]

    return None

geno["EFFECT"] = geno["INFO"].apply(get_effect)

# ============================================================
# KEEP ALL RELEVANT SNPs:
# missense + stop_gained + intergenic
# ============================================================
keep = geno["EFFECT"].str.contains(
    "missense|stop_gained|intergenic",
    na=False
)

geno_filt = geno[keep].copy()

print("\nNumber of retained SNPs:", geno_filt.shape[0])
print("\nRetained SNPs:")
for s in geno_filt["SNP_ID"].tolist():
    print(" ", s)

A_filt = A[geno_filt["SNP_ID"]].copy()

# ============================================================
# SKAT-LIKE SQUARED MAF-WEIGHTED GENE SCORE
# Original score: sum(w_j * G_ij)
# New SKAT-like score: sum((w_j * G_ij)^2)
# Since G is binary 0/1, this becomes sum(w_j^2 for present ALT SNPs)
# ============================================================
maf = A_filt.mean(axis=0)

weights = 1 / np.sqrt(maf * (1 - maf))
weights = weights.replace([np.inf, -np.inf], 0).fillna(0)

weighted_alt = A_filt * weights
gene_score = (weighted_alt ** 2).sum(axis=1)

df = pd.DataFrame({
    "Sample": gene_score.index,
    "Score": gene_score.values,
    "PC50": pc50_data.loc[gene_score.index, "PC50"].values
})

df["Group"] = np.where(df["PC50"] >= 5, "Delayed_PC50", "Low_PC50")
df["Binary"] = (df["PC50"] >= 5).astype(int)

df = df.dropna(subset=["PC50", "Score"]).copy()

print("\nAnalysis dataframe:")
print(df.to_string(index=False))

if df["Score"].nunique() <= 1:
    raise ValueError("Score has no variation across samples.")

# ============================================================
# CONTINUOUS ASSOCIATION: PC50 ~ SCORE
# ============================================================
X = sm.add_constant(df["Score"])
fit = sm.OLS(df["PC50"], X).fit()

pearson_r, pearson_p = pearsonr(df["Score"], df["PC50"])
spearman_r, spearman_p = spearmanr(df["Score"], df["PC50"])

# ============================================================
# GROUP TESTS
# ============================================================
low = df[df["Group"] == "Low_PC50"]["Score"]
delayed = df[df["Group"] == "Delayed_PC50"]["Score"]

welch_p = ttest_ind(low, delayed, equal_var=False).pvalue
mw_p = mannwhitneyu(low, delayed, alternative="two-sided").pvalue
wilcox_p = ranksums(low, delayed).pvalue

# ============================================================
# LOGISTIC REGRESSION
# ============================================================
try:
    logit = sm.Logit(df["Binary"], sm.add_constant(df["Score"])).fit(disp=0)
    logit_beta = logit.params.iloc[1]
    logit_p = logit.pvalues.iloc[1]
    odds_ratio = np.exp(logit_beta)
except Exception as e:
    logit_beta = np.nan
    logit_p = np.nan
    odds_ratio = np.nan
    print("\nLogistic regression failed:", e)

# ============================================================
# PRINT RESULTS
# ============================================================
print("\n" + "=" * 70)
print("SKAT-LIKE SQUARED MAF-WEIGHTED KIC1 SCORE USING PC50")
print("=" * 70)

print("\nWeights:")
print(weights.to_string())

print("\nSquared weights:")
print((weights ** 2).to_string())

print("\n--- Continuous association: PC50 ~ Score ---")
print(f"OLS beta = {fit.params.iloc[1]:.12f}")
print(f"OLS p = {fit.pvalues.iloc[1]:.12g}")
print(f"R^2 = {fit.rsquared:.12f}")
print(f"Pearson r = {pearson_r:.12f}, p = {pearson_p:.12g}")
print(f"Spearman rho = {spearman_r:.12f}, p = {spearman_p:.12g}")

print("\n--- Group tests: PC50 < 5 h vs PC50 >= 5 h ---")
print(f"Welch t-test p = {welch_p:.12g}")
print(f"Mann-Whitney p = {mw_p:.12g}")
print(f"Wilcoxon rank-sum p = {wilcox_p:.12g}")

print("\n--- Logistic regression: Delayed PC50 ~ Score ---")
print(f"Logistic beta = {logit_beta}")
print(f"Logistic p = {logit_p}")
print(f"Odds ratio = {odds_ratio}")

print("\n--- Group means ---")
print(f"Low PC50 mean score = {low.mean():.12f}")
print(f"Delayed PC50 mean score = {delayed.mean():.12f}")

print("\n--- Sample-level scores ---")
print(df.to_string(index=False))

# ============================================================
# PLOT 1: SCORE VS PC50 WITH SAMPLE LABELS
# ============================================================
plt.figure(figsize=(8, 6))

plt.scatter(df["Score"], df["PC50"], s=70)

xline = np.linspace(df["Score"].min(), df["Score"].max(), 100)
yline = fit.params.iloc[0] + fit.params.iloc[1] * xline
plt.plot(xline, yline)

plt.axhline(5, linestyle="--", linewidth=1)

for _, row in df.iterrows():
    plt.text(
        row["Score"] + 0.03,
        row["PC50"],
        row["Sample"],
        fontsize=7,
        alpha=0.8
    )

plt.xlabel("SKAT-like squared MAF-weighted KIC1 score")
plt.ylabel("PC50 (h)")
plt.title(f"KIC1 score vs PC50\nOLS p={fit.pvalues.iloc[1]:.2e}, R²={fit.rsquared:.2f}")

plt.tight_layout()
plt.show()

# ============================================================
# PLOT 2: VIOLIN + BOX + POINTS + SAMPLE LABELS
# ============================================================
plt.figure(figsize=(9, 6))

df_plot = df.copy()

plot_groups = [
    df_plot[df_plot["Group"] == "Low_PC50"],
    df_plot[df_plot["Group"] == "Delayed_PC50"]
]

pos = [1, 2]

plt.violinplot(
    [d["Score"] for d in plot_groups],
    positions=pos,
    showextrema=False
)

bp = plt.boxplot(
    [d["Score"] for d in plot_groups],
    positions=pos,
    widths=0.25,
    patch_artist=True,
    showfliers=False
)

for b in bp["boxes"]:
    b.set_facecolor("none")

rng = np.random.default_rng(0)

for i, d in enumerate(plot_groups):
    x_center = pos[i]
    xj = np.full(len(d), x_center, dtype=float) + rng.uniform(-0.09, 0.09, len(d))
    yj = d["Score"].values
    labels = d["Sample"].values

    plt.scatter(xj, yj, s=55)

    for x, y, label in zip(xj, yj, labels):
        plt.text(
            x + 0.025,
            y,
            label,
            fontsize=7,
            alpha=0.85
        )

plt.xticks(pos, ["PC50 < 5 h", "PC50 ≥ 5 h"])
plt.ylabel("SKAT-like squared MAF-weighted KIC1 score")
plt.title(f"KIC1 score by PC50 group\nWelch p={welch_p:.2e} | MW p={mw_p:.2e}")

plt.tight_layout()
plt.show()

# ============================================================
# PLOT 3: RANKED BAR PLOT WITH SAMPLE NAMES
# ============================================================
plt.figure(figsize=(10, 5))

df_sorted = df.sort_values("Score").reset_index(drop=True)
colors = ["grey" if g == "Low_PC50" else "black" for g in df_sorted["Group"]]

plt.bar(range(len(df_sorted)), df_sorted["Score"], color=colors)

plt.xticks(range(len(df_sorted)), df_sorted["Sample"], rotation=90)
plt.ylabel("SKAT-like squared MAF-weighted KIC1 score")
plt.xlabel("Samples")
plt.title("Ranked KIC1 scores: Low PC50 = grey, Delayed PC50 = black")

plt.tight_layout()
plt.show()

# ============================================================
# PERMUTATION TEST
# ============================================================
obs_diff = delayed.mean() - low.mean()

perm_diffs = []
rng = np.random.default_rng(1)

for _ in range(10000):
    shuffled = rng.permutation(df["Score"].values)
    perm_low = shuffled[:len(low)]
    perm_delayed = shuffled[len(low):]
    perm_diffs.append(perm_delayed.mean() - perm_low.mean())

p_perm = np.mean(np.abs(perm_diffs) >= abs(obs_diff))

print("\nPermutation p =", p_perm)

# ============================================================
# LOG-TRANSFORMED SCORE TEST
# ============================================================
df["Score_log"] = np.log1p(df["Score"])

low_log = df[df["Group"] == "Low_PC50"]["Score_log"]
delayed_log = df[df["Group"] == "Delayed_PC50"]["Score_log"]

print("\n--- Log-score group tests ---")
print("Welch p =", ttest_ind(low_log, delayed_log, equal_var=False).pvalue)
print("Mann-Whitney p =", mannwhitneyu(low_log, delayed_log, alternative="two-sided").pvalue)

# ============================================================
# STANDARDIZED LOGISTIC REGRESSION
# ============================================================
scaler = StandardScaler()
df["Score_z"] = scaler.fit_transform(df[["Score"]])

try:
    X_z = sm.add_constant(df["Score_z"])
    y_bin = df["Binary"]

    logit_z = sm.Logit(y_bin, X_z).fit(disp=0)

    print("\n" + "=" * 60)
    print("STANDARDIZED LOGISTIC REGRESSION")
    print("=" * 60)
    print("Coefficient =", logit_z.params["Score_z"])
    print("p-value =", logit_z.pvalues["Score_z"])
    print("Odds ratio =", np.exp(logit_z.params["Score_z"]))

    conf = logit_z.conf_int().loc["Score_z"]
    conf_odds = np.exp(conf)
    print("95% CI odds ratio =", tuple(conf_odds))

    print("\nFull summary:")
    print(logit_z.summary())

except Exception as e:
    print("\nStandardized logistic regression failed:", e)

# ============================================================
# PLOTTING BLOCK WITHOUT SAMPLE NAME LABELS
# ============================================================

# ------------------------------------------------------------
# PLOT 1: SCORE VS PC50, NO LABELS
# ------------------------------------------------------------
plt.figure(figsize=(8, 6))

plt.scatter(df["Score"], df["PC50"], s=70)

xline = np.linspace(df["Score"].min(), df["Score"].max(), 100)
yline = fit.params.iloc[0] + fit.params.iloc[1] * xline
plt.plot(xline, yline)

plt.axhline(5, linestyle="--", linewidth=1)

plt.xlabel("SKAT-like squared MAF-weighted KIC1 score")
plt.ylabel("PC50 (h)")
plt.title(f"KIC1 score vs PC50\nOLS p={fit.pvalues.iloc[1]:.2e}, R²={fit.rsquared:.2f}")

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# PLOT 2: VIOLIN + BOX + POINTS, NO LABELS
# ------------------------------------------------------------
plt.figure(figsize=(9, 6))

data = [low.values, delayed.values]
pos = [1, 2]

plt.violinplot(data, positions=pos, showextrema=False)

bp = plt.boxplot(
    data,
    positions=pos,
    widths=0.25,
    patch_artist=True,
    showfliers=False
)

for b in bp["boxes"]:
    b.set_facecolor("none")

rng = np.random.default_rng(0)

for i, arr in enumerate(data):
    xj = np.full(len(arr), pos[i], dtype=float) + rng.uniform(-0.09, 0.09, len(arr))
    plt.scatter(xj, arr, s=55)

plt.xticks(pos, ["PC50 < 5 h", "PC50 ≥ 5 h"])
plt.ylabel("SKAT-like squared MAF-weighted KIC1 score")
plt.title(f"KIC1 score by PC50 group\nWelch p={welch_p:.2e} | MW p={mw_p:.2e}")

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# PLOT 3: RANKED BAR PLOT, NO SAMPLE NAMES ON BARS
# ------------------------------------------------------------
plt.figure(figsize=(10, 5))

df_sorted = df.sort_values("Score").reset_index(drop=True)
colors = ["grey" if g == "Low_PC50" else "black" for g in df_sorted["Group"]]

plt.bar(range(len(df_sorted)), df_sorted["Score"], color=colors)

plt.ylabel("SKAT-like squared MAF-weighted KIC1 score")
plt.xlabel("Samples ranked")
plt.title("Ranked KIC1 scores: Low PC50 = grey, Delayed PC50 = black")

plt.tight_layout()
plt.show()


print("\nSNPs used in the KIC1 score:")
for i, row in geno_filt.iterrows():
    print(
        row["SNP_ID"],
        "| EFFECT:",
        row["EFFECT"],
        "| REF:",
        row["REF"],
        "| ALT:",
        row["ALT"]
    )


# ============================================================
# PENALIZED REGRESSION EXTENSION
# Append this block to your existing KIC1 analysis script.
# Requires: A_filt, weights, df (already constructed above)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import ElasticNetCV, ElasticNet
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import r2_score
from sklearn.utils import resample

# ------------------------------------------------------------
# STEP 1: Construct the weighted per-SNP feature matrix
# A_tilde_{ji} = w_i * A_{ji}
# Shape: (n_samples, m_snps) = (15, 7)
# ------------------------------------------------------------
A_tilde = A_filt.multiply(weights, axis=1)
A_tilde = A_tilde.loc[df["Sample"].values]   # align row order to df

X_ml = A_tilde.values                         # design matrix
y_ml = df["PC50"].values                      # continuous outcome
snp_names = A_tilde.columns.tolist()

print("\n" + "=" * 70)
print("PENALIZED REGRESSION (ELASTIC NET) ON KIC1 PER-SNP FEATURES")
print("=" * 70)
print(f"Design matrix shape: {X_ml.shape}")
print(f"Features (SNPs):     {len(snp_names)}")
print(f"Samples:             {X_ml.shape[0]}")

# ------------------------------------------------------------
# STEP 2 + 3: Elastic net with LOOCV over (alpha, l1_ratio)
# sklearn parameterization:
#   alpha       <-> lambda  (penalty strength)
#   l1_ratio    <-> alpha   (mixing: 1 = lasso, 0 = ridge)
# ------------------------------------------------------------
loo = LeaveOneOut()
l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]

enet_cv = ElasticNetCV(
    l1_ratio=l1_ratios,
    n_alphas=100,
    cv=loo,
    max_iter=20000,
    fit_intercept=True,
    random_state=0,
)

enet_cv.fit(X_ml, y_ml)

print("\n--- Selected hyperparameters (LOOCV) ---")
print(f"Optimal lambda (sklearn alpha): {enet_cv.alpha_:.6f}")
print(f"Optimal mixing l1_ratio:        {enet_cv.l1_ratio_:.2f}")

# ------------------------------------------------------------
# Refit at the optimal hyperparameters on full data
# ------------------------------------------------------------
enet_final = ElasticNet(
    alpha=enet_cv.alpha_,
    l1_ratio=enet_cv.l1_ratio_,
    max_iter=20000,
    fit_intercept=True,
)
enet_final.fit(X_ml, y_ml)

coef_table = pd.DataFrame({
    "SNP_ID": snp_names,
    "MB_weight": weights[snp_names].values,
    "Coefficient": enet_final.coef_,
    "Selected": enet_final.coef_ != 0,
})

print("\n--- Final coefficients ---")
print(coef_table.to_string(index=False))
print(f"\nNumber of SNPs with non-zero coefficient: "
      f"{int(coef_table['Selected'].sum())} / {len(snp_names)}")

# ------------------------------------------------------------
# STEP 4: Bootstrap stability selection
# pi_i = fraction of bootstrap resamples where SNP i is selected
# ------------------------------------------------------------
B = 1000
rng_boot = np.random.default_rng(42)
n_samples = X_ml.shape[0]
selection_matrix = np.zeros((B, len(snp_names)), dtype=int)

for b in range(B):
    idx = rng_boot.choice(n_samples, size=n_samples, replace=True)
    X_b, y_b = X_ml[idx], y_ml[idx]
    try:
        model_b = ElasticNet(
            alpha=enet_cv.alpha_,
            l1_ratio=enet_cv.l1_ratio_,
            max_iter=20000,
            fit_intercept=True,
        )
        model_b.fit(X_b, y_b)
        selection_matrix[b] = (model_b.coef_ != 0).astype(int)
    except Exception:
        selection_matrix[b] = 0

stability = selection_matrix.mean(axis=0)

stability_table = pd.DataFrame({
    "SNP_ID": snp_names,
    "MB_weight": weights[snp_names].values,
    "Final_coef": enet_final.coef_,
    "Stability_pi": stability,
    "Robust_pi>=0.8": stability >= 0.8,
}).sort_values("Stability_pi", ascending=False)

print("\n--- Stability selection (B = 1000 bootstrap resamples) ---")
print(stability_table.to_string(index=False))

# ------------------------------------------------------------
# STEP 5: Permutation test on full-model R^2
# ------------------------------------------------------------
y_pred_obs = enet_final.predict(X_ml)
r2_obs = r2_score(y_ml, y_pred_obs)

n_perm = 1000
rng_perm = np.random.default_rng(7)
r2_null = np.zeros(n_perm)

for b in range(n_perm):
    y_perm = rng_perm.permutation(y_ml)
    model_p = ElasticNet(
        alpha=enet_cv.alpha_,
        l1_ratio=enet_cv.l1_ratio_,
        max_iter=20000,
        fit_intercept=True,
    )
    model_p.fit(X_ml, y_perm)
    r2_null[b] = r2_score(y_perm, model_p.predict(X_ml))

p_perm_r2 = (1 + np.sum(r2_null >= r2_obs)) / (1 + n_perm)

print("\n--- Permutation test of model R^2 ---")
print(f"Observed in-sample R^2: {r2_obs:.4f}")
print(f"Null R^2 mean:          {r2_null.mean():.4f}")
print(f"Null R^2 95th pct:      {np.quantile(r2_null, 0.95):.4f}")
print(f"Permutation p-value:    {p_perm_r2:.4f}")

# ------------------------------------------------------------
# PLOT: Stability scores per SNP
# ------------------------------------------------------------
plt.figure(figsize=(9, 5))
order = np.argsort(stability)[::-1]
plt.bar(range(len(snp_names)),
        stability[order],
        color=["black" if stability[i] >= 0.8 else "grey" for i in order])
plt.axhline(0.8, linestyle="--", linewidth=1)
plt.xticks(range(len(snp_names)),
           [snp_names[i] for i in order],
           rotation=45, ha="right", fontsize=8)
plt.ylabel("Stability score π_i")
plt.ylim(0, 1.05)
plt.title("Bootstrap stability selection of KIC1 SNPs (Elastic Net, B=1000)")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# PLOT: Observed vs null R^2 distribution
# ------------------------------------------------------------
plt.figure(figsize=(8, 5))
plt.hist(r2_null, bins=40, color="grey", edgecolor="white")
plt.axvline(r2_obs, color="black", linewidth=2)
plt.xlabel("R² (in-sample)")
plt.ylabel("Frequency")
plt.title(f"Permutation null vs observed R²\nObserved={r2_obs:.3f}, p={p_perm_r2:.3f}")
plt.tight_layout()
plt.show()

Association of KIC1 using RSA

In [ ]:
import re
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

from scipy.stats import ttest_ind, mannwhitneyu, pearsonr, spearmanr, ranksums
from sklearn.preprocessing import StandardScaler

# ============================================================
# FILE
# ============================================================
geno = pd.read_csv("KIC_1_vcf_subset_BD - Sheet1.csv")
geno.columns = [str(c).strip() for c in geno.columns]

# ============================================================
# RSA VALUES
# RSA >= 1 = resistant
# RSA < 1 = sensitive
# ============================================================
rsa = pd.DataFrame({
    "Sample": [
        "I-001-clone", "I-003-clone", "I-004", "I-011", "I-016",
        "I-018", "I-020", "I-029", "I-026", "I-033",
        "I-039", "I-041", "I-040", "I-027", "O-024",
        "I-008", "I-002", "O-014", "O-009", "O-032",
        "O-038", "I-019"
    ],
    "RSA": [
        0.6635725467, 2.007394647, 0, 0.6057562733, 4.634666667,
        0.7466666667, 1.76, 6.443333333, 0.6533333333, 1.889666667,
        4.891666667, 1.358666667, 0.7466666667, 0.5593333333, 1.73,
        1.535666667, 2.563333333, 2.126666667, 0.42, 0.783,
        1.371666667, 0.4633333333
    ]
})

rsa["Sample"] = rsa["Sample"].astype(str).str.strip()
rsa["RSA"] = pd.to_numeric(rsa["RSA"], errors="coerce")
rsa = rsa.dropna()

rsa["Group"] = np.where(rsa["RSA"] >= 1, "Resistant_RSA", "Sensitive_RSA")
rsa["Binary"] = (rsa["RSA"] >= 1).astype(int)

print("RSA phenotype table:")
print(rsa.to_string(index=False))

print("\nRSA group counts:")
print(rsa["Group"].value_counts())

# ============================================================
# MATCH SAMPLES
# ============================================================
if "FORMAT" not in geno.columns:
    raise ValueError("FORMAT column not found in genotype file.")

format_idx = list(geno.columns).index("FORMAT")
samples = list(geno.columns[format_idx + 1:])
samples = [str(s).strip() for s in samples]

common = sorted(set(samples).intersection(set(rsa["Sample"])))

print("\nSamples in VCF:", len(samples))
print("Samples with RSA:", len(rsa))
print("Matched samples:", len(common))
print(common)

if len(common) == 0:
    raise ValueError("No matching sample names between genotype file and RSA table.")

rsa_data = rsa.set_index("Sample").loc[common]

# ============================================================
# SNP IDs
# ============================================================
chrom_col = "#CHROM" if "#CHROM" in geno.columns else "CHROM"

geno["POS"] = pd.to_numeric(geno["POS"], errors="coerce")
geno = geno.dropna(subset=["POS"]).copy()
geno["POS"] = geno["POS"].astype(int)

geno["SNP_ID"] = geno.apply(
    lambda r: f"{r[chrom_col]}:{int(r['POS'])}:{r['REF']}>{r['ALT']}",
    axis=1
)

# ============================================================
# ALT MATRIX
# ============================================================
def alt_present(x):
    if pd.isna(x):
        return 0.0

    gt = str(x).split(":")[0]

    if gt in ["./.", ".|.", "."]:
        return 0.0

    alleles = re.split(r"[\/|]", gt)
    return 1.0 if "1" in alleles else 0.0

A = pd.DataFrame(
    {s: geno[s].map(alt_present).values for s in common},
    index=geno["SNP_ID"]
).T

# ============================================================
# PARSE EFFECT FROM ANN
# ============================================================
def get_effect(info):
    if pd.isna(info):
        return None

    for field in str(info).split(";"):
        if field.startswith("ANN="):
            ann = field[4:].split(",")[0]
            parts = ann.split("|")
            if len(parts) > 1:
                return parts[1]

    return None

geno["EFFECT"] = geno["INFO"].apply(get_effect)

# ============================================================
# KEEP RELEVANT SNPs
# ============================================================
keep = geno["EFFECT"].str.contains(
    "missense|stop_gained|intergenic",
    na=False
)

geno_filt = geno[keep].copy()

print("\nNumber of retained SNPs:", geno_filt.shape[0])
print("\nRetained SNPs:")
for s in geno_filt["SNP_ID"].tolist():
    print(" ", s)

A_filt = A[geno_filt["SNP_ID"]].copy()

# ============================================================
# SKAT-LIKE SQUARED MAF-WEIGHTED GENE SCORE
# ============================================================
maf = A_filt.mean(axis=0)

weights = 1 / np.sqrt(maf * (1 - maf))
weights = weights.replace([np.inf, -np.inf], 0).fillna(0)

weighted_alt = A_filt * weights
gene_score = (weighted_alt ** 2).sum(axis=1)

df = pd.DataFrame({
    "Sample": gene_score.index,
    "Score": gene_score.values,
    "RSA": rsa_data.loc[gene_score.index, "RSA"].values,
    "Group": rsa_data.loc[gene_score.index, "Group"].values,
    "Binary": rsa_data.loc[gene_score.index, "Binary"].values
})

df = df.dropna(subset=["RSA", "Score"]).copy()

print("\nAnalysis dataframe:")
print(df.to_string(index=False))

print("\nMatched sensitive/resistant counts:")
print(df["Group"].value_counts())

if df["Score"].nunique() <= 1:
    raise ValueError("Score has no variation across samples.")

# ============================================================
# CONTINUOUS ASSOCIATION: RSA ~ SCORE
# ============================================================
X = sm.add_constant(df["Score"])
fit = sm.OLS(df["RSA"], X).fit()

pearson_r, pearson_p = pearsonr(df["Score"], df["RSA"])
spearman_r, spearman_p = spearmanr(df["Score"], df["RSA"])

# ============================================================
# GROUP TESTS
# ============================================================
sensitive = df[df["Group"] == "Sensitive_RSA"]["Score"]
resistant = df[df["Group"] == "Resistant_RSA"]["Score"]

welch_p = ttest_ind(sensitive, resistant, equal_var=False).pvalue
mw_p = mannwhitneyu(sensitive, resistant, alternative="two-sided").pvalue
wilcox_p = ranksums(sensitive, resistant).pvalue

# ============================================================
# LOGISTIC REGRESSION
# ============================================================
try:
    logit = sm.Logit(df["Binary"], sm.add_constant(df["Score"])).fit(disp=0)
    logit_beta = logit.params.iloc[1]
    logit_p = logit.pvalues.iloc[1]
    odds_ratio = np.exp(logit_beta)
except Exception as e:
    logit_beta = np.nan
    logit_p = np.nan
    odds_ratio = np.nan
    print("\nLogistic regression failed:", e)

# ============================================================
# PRINT RESULTS
# ============================================================
print("\n" + "=" * 70)
print("SKAT-LIKE SQUARED MAF-WEIGHTED KIC1 SCORE USING RSA")
print("RSA >= 1 = Resistant; RSA < 1 = Sensitive")
print("=" * 70)

print("\nWeights:")
print(weights.to_string())

print("\nSquared weights:")
print((weights ** 2).to_string())

print("\n--- Continuous association: RSA ~ Score ---")
print(f"OLS beta = {fit.params.iloc[1]:.12f}")
print(f"OLS p = {fit.pvalues.iloc[1]:.12g}")
print(f"R^2 = {fit.rsquared:.12f}")
print(f"Pearson r = {pearson_r:.12f}, p = {pearson_p:.12g}")
print(f"Spearman rho = {spearman_r:.12f}, p = {spearman_p:.12g}")

print("\n--- Group tests: Sensitive RSA < 1 vs Resistant RSA >= 1 ---")
print(f"Welch t-test p = {welch_p:.12g}")
print(f"Mann-Whitney p = {mw_p:.12g}")
print(f"Wilcoxon rank-sum p = {wilcox_p:.12g}")

print("\n--- Logistic regression: Resistant RSA ~ Score ---")
print(f"Logistic beta = {logit_beta}")
print(f"Logistic p = {logit_p}")
print(f"Odds ratio = {odds_ratio}")

print("\n--- Group means ---")
print(f"Sensitive RSA mean score = {sensitive.mean():.12f}")
print(f"Resistant RSA mean score = {resistant.mean():.12f}")

print("\n--- Sample-level scores ---")
print(df.to_string(index=False))

# ============================================================
# PLOT 1: SCORE VS RSA WITH SAMPLE LABELS
# ============================================================
plt.figure(figsize=(8, 6))

plt.scatter(df["Score"], df["RSA"], s=70)

xline = np.linspace(df["Score"].min(), df["Score"].max(), 100)
yline = fit.params.iloc[0] + fit.params.iloc[1] * xline
plt.plot(xline, yline)

plt.axhline(1, linestyle="--", linewidth=1)

for _, row in df.iterrows():
    plt.text(
        row["Score"] + 0.03,
        row["RSA"],
        row["Sample"],
        fontsize=7,
        alpha=0.8
    )

plt.xlabel("SKAT-like squared MAF-weighted KIC1 score")
plt.ylabel("RSA")
plt.title(f"KIC1 score vs RSA\nOLS p={fit.pvalues.iloc[1]:.2e}, R²={fit.rsquared:.2f}")

plt.tight_layout()
plt.show()

# ============================================================
# PLOT 2: VIOLIN + BOX + POINTS + SAMPLE LABELS
# ============================================================
plt.figure(figsize=(9, 6))

plot_groups = [
    df[df["Group"] == "Sensitive_RSA"],
    df[df["Group"] == "Resistant_RSA"]
]

pos = [1, 2]

plt.violinplot(
    [d["Score"] for d in plot_groups],
    positions=pos,
    showextrema=False
)

bp = plt.boxplot(
    [d["Score"] for d in plot_groups],
    positions=pos,
    widths=0.25,
    patch_artist=True,
    showfliers=False
)

for b in bp["boxes"]:
    b.set_facecolor("none")

rng = np.random.default_rng(0)

for i, d in enumerate(plot_groups):
    x_center = pos[i]
    xj = np.full(len(d), x_center, dtype=float) + rng.uniform(-0.09, 0.09, len(d))
    yj = d["Score"].values
    labels = d["Sample"].values

    plt.scatter(xj, yj, s=55)

    for x, y, label in zip(xj, yj, labels):
        plt.text(
            x + 0.025,
            y,
            label,
            fontsize=7,
            alpha=0.85
        )

plt.xticks(pos, ["RSA < 1\nSensitive", "RSA ≥ 1\nResistant"])
plt.ylabel("SKAT-like squared MAF-weighted KIC1 score")
plt.title(f"KIC1 score by RSA group\nWelch p={welch_p:.2e} | MW p={mw_p:.2e}")

plt.tight_layout()
plt.show()

# ============================================================
# PLOT 3: RANKED BAR PLOT WITH SAMPLE NAMES
# ============================================================
plt.figure(figsize=(10, 5))

df_sorted = df.sort_values("Score").reset_index(drop=True)
colors = ["grey" if g == "Sensitive_RSA" else "black" for g in df_sorted["Group"]]

plt.bar(range(len(df_sorted)), df_sorted["Score"], color=colors)

plt.xticks(range(len(df_sorted)), df_sorted["Sample"], rotation=90)
plt.ylabel("SKAT-like squared MAF-weighted KIC1 score")
plt.xlabel("Samples")
plt.title("Ranked KIC1 scores: Sensitive RSA = grey, Resistant RSA = black")

plt.tight_layout()
plt.show()

# ============================================================
# PERMUTATION TEST
# ============================================================
obs_diff = resistant.mean() - sensitive.mean()

perm_diffs = []
rng = np.random.default_rng(1)

for _ in range(10000):
    shuffled = rng.permutation(df["Score"].values)
    perm_sensitive = shuffled[:len(sensitive)]
    perm_resistant = shuffled[len(sensitive):]
    perm_diffs.append(perm_resistant.mean() - perm_sensitive.mean())

p_perm = np.mean(np.abs(perm_diffs) >= abs(obs_diff))

print("\nPermutation p =", p_perm)

# ============================================================
# LOG-TRANSFORMED SCORE TEST
# ============================================================
df["Score_log"] = np.log1p(df["Score"])

sensitive_log = df[df["Group"] == "Sensitive_RSA"]["Score_log"]
resistant_log = df[df["Group"] == "Resistant_RSA"]["Score_log"]

print("\n--- Log-score group tests ---")
print("Welch p =", ttest_ind(sensitive_log, resistant_log, equal_var=False).pvalue)
print("Mann-Whitney p =", mannwhitneyu(sensitive_log, resistant_log, alternative="two-sided").pvalue)

# ============================================================
# STANDARDIZED LOGISTIC REGRESSION
# ============================================================
scaler = StandardScaler()
df["Score_z"] = scaler.fit_transform(df[["Score"]])

try:
    X_z = sm.add_constant(df["Score_z"])
    y_bin = df["Binary"]

    logit_z = sm.Logit(y_bin, X_z).fit(disp=0)

    print("\n" + "=" * 60)
    print("STANDARDIZED LOGISTIC REGRESSION")
    print("=" * 60)
    print("Coefficient =", logit_z.params["Score_z"])
    print("p-value =", logit_z.pvalues["Score_z"])
    print("Odds ratio =", np.exp(logit_z.params["Score_z"]))

    conf = logit_z.conf_int().loc["Score_z"]
    conf_odds = np.exp(conf)
    print("95% CI odds ratio =", tuple(conf_odds))

    print("\nFull summary:")
    print(logit_z.summary())

except Exception as e:
    print("\nStandardized logistic regression failed:", e)

# ============================================================
# PLOTTING BLOCK WITHOUT SAMPLE NAME LABELS
# ============================================================

# ------------------------------------------------------------
# PLOT 1: SCORE VS RSA, NO LABELS
# ------------------------------------------------------------
plt.figure(figsize=(8, 6))

plt.scatter(df["Score"], df["RSA"], s=70)

xline = np.linspace(df["Score"].min(), df["Score"].max(), 100)
yline = fit.params.iloc[0] + fit.params.iloc[1] * xline
plt.plot(xline, yline)

plt.axhline(1, linestyle="--", linewidth=1)

plt.xlabel("SKAT-like squared MAF-weighted KIC1 score")
plt.ylabel("RSA")
plt.title(f"KIC1 score vs RSA\nOLS p={fit.pvalues.iloc[1]:.2e}, R²={fit.rsquared:.2f}")

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# PLOT 2: VIOLIN + BOX + POINTS, NO LABELS
# ------------------------------------------------------------
plt.figure(figsize=(9, 6))

data = [sensitive.values, resistant.values]
pos = [1, 2]

plt.violinplot(data, positions=pos, showextrema=False)

bp = plt.boxplot(
    data,
    positions=pos,
    widths=0.25,
    patch_artist=True,
    showfliers=False
)

for b in bp["boxes"]:
    b.set_facecolor("none")

rng = np.random.default_rng(0)

for i, arr in enumerate(data):
    xj = np.full(len(arr), pos[i], dtype=float) + rng.uniform(-0.09, 0.09, len(arr))
    plt.scatter(xj, arr, s=55)

plt.xticks(pos, ["RSA < 1\nSensitive", "RSA ≥ 1\nResistant"])
plt.ylabel("SKAT-like squared MAF-weighted KIC1 score")
plt.title(f"KIC1 score by RSA group\nWelch p={welch_p:.2e} | MW p={mw_p:.2e}")

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# PLOT 3: RANKED BAR PLOT, NO SAMPLE NAMES ON BARS
# ------------------------------------------------------------
plt.figure(figsize=(10, 5))

df_sorted = df.sort_values("Score").reset_index(drop=True)
colors = ["grey" if g == "Sensitive_RSA" else "black" for g in df_sorted["Group"]]

plt.bar(range(len(df_sorted)), df_sorted["Score"], color=colors)

plt.ylabel("SKAT-like squared MAF-weighted KIC1 score")
plt.xlabel("Samples ranked")
plt.title("Ranked KIC1 scores: Sensitive RSA = grey, Resistant RSA = black")

plt.tight_layout()
plt.show()

print("\nSNPs used in the KIC1 score:")
for i, row in geno_filt.iterrows():
    print(
        row["SNP_ID"],
        "| EFFECT:",
        row["EFFECT"],
        "| REF:",
        row["REF"],
        "| ALT:",
        row["ALT"]
    )

In [ ]:
import re
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

from scipy.stats import ttest_ind, mannwhitneyu, pearsonr, spearmanr, ranksums
from sklearn.preprocessing import StandardScaler

# ============================================================
# FILE
# ============================================================
geno = pd.read_csv("KIC_1_vcf_subset_BD - Sheet1.csv")
geno.columns = [str(c).strip() for c in geno.columns]

# ============================================================
# RSA VALUES
# RSA >= 1 = resistant
# RSA < 1 = sensitive
# ============================================================
rsa = pd.DataFrame({
    "Sample": [
        "I-001-clone", "I-003-clone", "I-004", "I-011", "I-016",
        "I-018", "I-020", "I-029", "I-026", "I-033",
        "I-039", "I-041", "I-040", "I-027", "O-024",
        "I-008", "I-002", "O-014", "O-009", "O-032",
        "O-038", "I-019"
    ],
    "RSA": [
        0.6635725467, 2.007394647, 0, 0.6057562733, 4.634666667,
        0.7466666667, 1.76, 6.443333333, 0.6533333333, 1.889666667,
        4.891666667, 1.358666667, 0.7466666667, 0.5593333333, 1.73,
        1.535666667, 2.563333333, 2.126666667, 0.42, 0.783,
        1.371666667, 0.4633333333
    ]
})

rsa["Sample"] = rsa["Sample"].astype(str).str.strip()
rsa["RSA"] = pd.to_numeric(rsa["RSA"], errors="coerce")
rsa = rsa.dropna()

rsa["Group"] = np.where(rsa["RSA"] >= 1, "Resistant_RSA", "Sensitive_RSA")
rsa["Binary"] = (rsa["RSA"] >= 1).astype(int)

print("RSA phenotype table:")
print(rsa.to_string(index=False))

print("\nRSA group counts:")
print(rsa["Group"].value_counts())

# ============================================================
# MATCH SAMPLES
# ============================================================
if "FORMAT" not in geno.columns:
    raise ValueError("FORMAT column not found in genotype file.")

format_idx = list(geno.columns).index("FORMAT")
samples = list(geno.columns[format_idx + 1:])
samples = [str(s).strip() for s in samples]

common = sorted(set(samples).intersection(set(rsa["Sample"])))

print("\nSamples in VCF:", len(samples))
print("Samples with RSA:", len(rsa))
print("Matched samples:", len(common))
print(common)

if len(common) == 0:
    raise ValueError("No matching sample names between genotype file and RSA table.")

rsa_data = rsa.set_index("Sample").loc[common]

# ============================================================
# SNP IDs
# ============================================================
chrom_col = "#CHROM" if "#CHROM" in geno.columns else "CHROM"

geno["POS"] = pd.to_numeric(geno["POS"], errors="coerce")
geno = geno.dropna(subset=["POS"]).copy()
geno["POS"] = geno["POS"].astype(int)

geno["SNP_ID"] = geno.apply(
    lambda r: f"{r[chrom_col]}:{int(r['POS'])}:{r['REF']}>{r['ALT']}",
    axis=1
)

# ============================================================
# ALT MATRIX
# ============================================================
def alt_present(x):
    if pd.isna(x):
        return 0.0

    gt = str(x).split(":")[0]

    if gt in ["./.", ".|.", "."]:
        return 0.0

    alleles = re.split(r"[\/|]", gt)
    return 1.0 if "1" in alleles else 0.0

A = pd.DataFrame(
    {s: geno[s].map(alt_present).values for s in common},
    index=geno["SNP_ID"]
).T

# ============================================================
# PARSE EFFECT FROM ANN
# ============================================================
def get_effect(info):
    if pd.isna(info):
        return None

    for field in str(info).split(";"):
        if field.startswith("ANN="):
            ann = field[4:].split(",")[0]
            parts = ann.split("|")
            if len(parts) > 1:
                return parts[1]

    return None

geno["EFFECT"] = geno["INFO"].apply(get_effect)

# ============================================================
# KEEP RELEVANT SNPs
# ============================================================
keep = geno["EFFECT"].str.contains(
    "missense|stop_gained|intergenic",
    na=False
)

geno_filt = geno[keep].copy()

print("\nNumber of retained SNPs:", geno_filt.shape[0])
print("\nRetained SNPs:")
for s in geno_filt["SNP_ID"].tolist():
    print(" ", s)

A_filt = A[geno_filt["SNP_ID"]].copy()

# ============================================================
# SKAT-LIKE SQUARED MAF-WEIGHTED GENE SCORE
# ============================================================
maf = A_filt.mean(axis=0)

weights = 1 / np.sqrt(maf * (1 - maf))
weights = weights.replace([np.inf, -np.inf], 0).fillna(0)

weighted_alt = A_filt * weights
gene_score = (weighted_alt ** 2).sum(axis=1)

df = pd.DataFrame({
    "Sample": gene_score.index,
    "Score": gene_score.values,
    "RSA": rsa_data.loc[gene_score.index, "RSA"].values,
    "Group": rsa_data.loc[gene_score.index, "Group"].values,
    "Binary": rsa_data.loc[gene_score.index, "Binary"].values
})

df = df.dropna(subset=["RSA", "Score"]).copy()

print("\nAnalysis dataframe:")
print(df.to_string(index=False))

print("\nMatched sensitive/resistant counts:")
print(df["Group"].value_counts())

if df["Score"].nunique() <= 1:
    raise ValueError("Score has no variation across samples.")

# ============================================================
# CONTINUOUS ASSOCIATION: RSA ~ SCORE
# ============================================================
X = sm.add_constant(df["Score"])
fit = sm.OLS(df["RSA"], X).fit()

pearson_r, pearson_p = pearsonr(df["Score"], df["RSA"])
spearman_r, spearman_p = spearmanr(df["Score"], df["RSA"])

# ============================================================
# GROUP TESTS
# ============================================================
sensitive = df[df["Group"] == "Sensitive_RSA"]["Score"]
resistant = df[df["Group"] == "Resistant_RSA"]["Score"]

welch_p = ttest_ind(sensitive, resistant, equal_var=False).pvalue
mw_p = mannwhitneyu(sensitive, resistant, alternative="two-sided").pvalue
wilcox_p = ranksums(sensitive, resistant).pvalue

# ============================================================
# LOGISTIC REGRESSION
# ============================================================
try:
    logit = sm.Logit(df["Binary"], sm.add_constant(df["Score"])).fit(disp=0)
    logit_beta = logit.params.iloc[1]
    logit_p = logit.pvalues.iloc[1]
    odds_ratio = np.exp(logit_beta)
except Exception as e:
    logit_beta = np.nan
    logit_p = np.nan
    odds_ratio = np.nan
    print("\nLogistic regression failed:", e)

# ============================================================
# PRINT RESULTS
# ============================================================
print("\n" + "=" * 70)
print("SKAT-LIKE SQUARED MAF-WEIGHTED KIC1 SCORE USING RSA")
print("RSA >= 1 = Resistant; RSA < 1 = Sensitive")
print("=" * 70)

print("\nWeights:")
print(weights.to_string())

print("\nSquared weights:")
print((weights ** 2).to_string())

print("\n--- Continuous association: RSA ~ Score ---")
print(f"OLS beta = {fit.params.iloc[1]:.12f}")
print(f"OLS p = {fit.pvalues.iloc[1]:.12g}")
print(f"R^2 = {fit.rsquared:.12f}")
print(f"Pearson r = {pearson_r:.12f}, p = {pearson_p:.12g}")
print(f"Spearman rho = {spearman_r:.12f}, p = {spearman_p:.12g}")

print("\n--- Group tests: Sensitive RSA < 1 vs Resistant RSA >= 1 ---")
print(f"Welch t-test p = {welch_p:.12g}")
print(f"Mann-Whitney p = {mw_p:.12g}")
print(f"Wilcoxon rank-sum p = {wilcox_p:.12g}")

print("\n--- Logistic regression: Resistant RSA ~ Score ---")
print(f"Logistic beta = {logit_beta}")
print(f"Logistic p = {logit_p}")
print(f"Odds ratio = {odds_ratio}")

print("\n--- Group means ---")
print(f"Sensitive RSA mean score = {sensitive.mean():.12f}")
print(f"Resistant RSA mean score = {resistant.mean():.12f}")

print("\n--- Sample-level scores ---")
print(df.to_string(index=False))

# ============================================================
# PLOT 1: SCORE VS RSA WITH SAMPLE LABELS
# ============================================================
plt.figure(figsize=(8, 6))

plt.scatter(df["Score"], df["RSA"], s=70)

xline = np.linspace(df["Score"].min(), df["Score"].max(), 100)
yline = fit.params.iloc[0] + fit.params.iloc[1] * xline
plt.plot(xline, yline)

plt.axhline(1, linestyle="--", linewidth=1)

for _, row in df.iterrows():
    plt.text(
        row["Score"] + 0.03,
        row["RSA"],
        row["Sample"],
        fontsize=7,
        alpha=0.8
    )

plt.xlabel("SKAT-like squared MAF-weighted KIC1 score")
plt.ylabel("RSA")
plt.title(f"KIC1 score vs RSA\nOLS p={fit.pvalues.iloc[1]:.2e}, R²={fit.rsquared:.2f}")

plt.tight_layout()
plt.show()

# ============================================================
# PLOT 2: VIOLIN + BOX + POINTS + SAMPLE LABELS
# ============================================================
plt.figure(figsize=(9, 6))

plot_groups = [
    df[df["Group"] == "Sensitive_RSA"],
    df[df["Group"] == "Resistant_RSA"]
]

pos = [1, 2]

plt.violinplot(
    [d["Score"] for d in plot_groups],
    positions=pos,
    showextrema=False
)

bp = plt.boxplot(
    [d["Score"] for d in plot_groups],
    positions=pos,
    widths=0.25,
    patch_artist=True,
    showfliers=False
)

for b in bp["boxes"]:
    b.set_facecolor("none")

rng = np.random.default_rng(0)

for i, d in enumerate(plot_groups):
    x_center = pos[i]
    xj = np.full(len(d), x_center, dtype=float) + rng.uniform(-0.09, 0.09, len(d))
    yj = d["Score"].values
    labels = d["Sample"].values

    plt.scatter(xj, yj, s=55)

    for x, y, label in zip(xj, yj, labels):
        plt.text(
            x + 0.025,
            y,
            label,
            fontsize=7,
            alpha=0.85
        )

plt.xticks(pos, ["RSA < 1\nSensitive", "RSA ≥ 1\nResistant"])
plt.ylabel("SKAT-like squared MAF-weighted KIC1 score")
plt.title(f"KIC1 score by RSA group\nWelch p={welch_p:.2e} | MW p={mw_p:.2e}")

plt.tight_layout()
plt.show()

# ============================================================
# PLOT 3: RANKED BAR PLOT WITH SAMPLE NAMES
# ============================================================
plt.figure(figsize=(10, 5))

df_sorted = df.sort_values("Score").reset_index(drop=True)
colors = ["grey" if g == "Sensitive_RSA" else "black" for g in df_sorted["Group"]]

plt.bar(range(len(df_sorted)), df_sorted["Score"], color=colors)

plt.xticks(range(len(df_sorted)), df_sorted["Sample"], rotation=90)
plt.ylabel("SKAT-like squared MAF-weighted KIC1 score")
plt.xlabel("Samples")
plt.title("Ranked KIC1 scores: Sensitive RSA = grey, Resistant RSA = black")

plt.tight_layout()
plt.show()

# ============================================================
# PERMUTATION TEST
# ============================================================
obs_diff = resistant.mean() - sensitive.mean()

perm_diffs = []
rng = np.random.default_rng(1)

for _ in range(10000):
    shuffled = rng.permutation(df["Score"].values)
    perm_sensitive = shuffled[:len(sensitive)]
    perm_resistant = shuffled[len(sensitive):]
    perm_diffs.append(perm_resistant.mean() - perm_sensitive.mean())

p_perm = np.mean(np.abs(perm_diffs) >= abs(obs_diff))

print("\nPermutation p =", p_perm)

# ============================================================
# LOG-TRANSFORMED SCORE TEST
# ============================================================
df["Score_log"] = np.log1p(df["Score"])

sensitive_log = df[df["Group"] == "Sensitive_RSA"]["Score_log"]
resistant_log = df[df["Group"] == "Resistant_RSA"]["Score_log"]

print("\n--- Log-score group tests ---")
print("Welch p =", ttest_ind(sensitive_log, resistant_log, equal_var=False).pvalue)
print("Mann-Whitney p =", mannwhitneyu(sensitive_log, resistant_log, alternative="two-sided").pvalue)

# ============================================================
# STANDARDIZED LOGISTIC REGRESSION
# ============================================================
scaler = StandardScaler()
df["Score_z"] = scaler.fit_transform(df[["Score"]])

try:
    X_z = sm.add_constant(df["Score_z"])
    y_bin = df["Binary"]

    logit_z = sm.Logit(y_bin, X_z).fit(disp=0)

    print("\n" + "=" * 60)
    print("STANDARDIZED LOGISTIC REGRESSION")
    print("=" * 60)
    print("Coefficient =", logit_z.params["Score_z"])
    print("p-value =", logit_z.pvalues["Score_z"])
    print("Odds ratio =", np.exp(logit_z.params["Score_z"]))

    conf = logit_z.conf_int().loc["Score_z"]
    conf_odds = np.exp(conf)
    print("95% CI odds ratio =", tuple(conf_odds))

    print("\nFull summary:")
    print(logit_z.summary())

except Exception as e:
    print("\nStandardized logistic regression failed:", e)

# ============================================================
# PLOTTING BLOCK WITHOUT SAMPLE NAME LABELS
# ============================================================

# ------------------------------------------------------------
# PLOT 1: SCORE VS RSA, NO LABELS
# ------------------------------------------------------------
plt.figure(figsize=(8, 6))

plt.scatter(df["Score"], df["RSA"], s=70)

xline = np.linspace(df["Score"].min(), df["Score"].max(), 100)
yline = fit.params.iloc[0] + fit.params.iloc[1] * xline
plt.plot(xline, yline)

plt.axhline(1, linestyle="--", linewidth=1)

plt.xlabel("SKAT-like squared MAF-weighted KIC1 score")
plt.ylabel("RSA")
plt.title(f"KIC1 score vs RSA\nOLS p={fit.pvalues.iloc[1]:.2e}, R²={fit.rsquared:.2f}")

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# PLOT 2: VIOLIN + BOX + POINTS, NO LABELS
# ------------------------------------------------------------
plt.figure(figsize=(9, 6))

data = [sensitive.values, resistant.values]
pos = [1, 2]

plt.violinplot(data, positions=pos, showextrema=False)

bp = plt.boxplot(
    data,
    positions=pos,
    widths=0.25,
    patch_artist=True,
    showfliers=False
)

for b in bp["boxes"]:
    b.set_facecolor("none")

rng = np.random.default_rng(0)

for i, arr in enumerate(data):
    xj = np.full(len(arr), pos[i], dtype=float) + rng.uniform(-0.09, 0.09, len(arr))
    plt.scatter(xj, arr, s=55)

plt.xticks(pos, ["RSA < 1\nSensitive", "RSA ≥ 1\nResistant"])
plt.ylabel("SKAT-like squared MAF-weighted KIC1 score")
plt.title(f"KIC1 score by RSA group\nWelch p={welch_p:.2e} | MW p={mw_p:.2e}")

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# PLOT 3: RANKED BAR PLOT, NO SAMPLE NAMES ON BARS
# ------------------------------------------------------------
plt.figure(figsize=(10, 5))

df_sorted = df.sort_values("Score").reset_index(drop=True)
colors = ["grey" if g == "Sensitive_RSA" else "black" for g in df_sorted["Group"]]

plt.bar(range(len(df_sorted)), df_sorted["Score"], color=colors)

plt.ylabel("SKAT-like squared MAF-weighted KIC1 score")
plt.xlabel("Samples ranked")
plt.title("Ranked KIC1 scores: Sensitive RSA = grey, Resistant RSA = black")

plt.tight_layout()
plt.show()

print("\nSNPs used in the KIC1 score:")
for i, row in geno_filt.iterrows():
    print(
        row["SNP_ID"],
        "| EFFECT:",
        row["EFFECT"],
        "| REF:",
        row["REF"],
        "| ALT:",
        row["ALT"]
    )


# ============================================================
# PENALIZED REGRESSION EXTENSION — RSA PHENOTYPE
# Append this block to your existing KIC1 RSA analysis script.
# Requires: A_filt, weights, df (already constructed above)
# df must contain columns: Sample, Score, RSA, Group, Binary
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import ElasticNetCV, ElasticNet
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import r2_score

# ------------------------------------------------------------
# STEP 1: Construct the weighted per-SNP feature matrix
# A_tilde_{ji} = w_i * A_{ji}
# Shape: (n_samples, m_snps)
# ------------------------------------------------------------
A_tilde = A_filt.multiply(weights, axis=1)
A_tilde = A_tilde.loc[df["Sample"].values]   # align row order to df

X_ml = A_tilde.values                         # design matrix
y_ml = df["RSA"].values                       # continuous outcome (RSA)
snp_names = A_tilde.columns.tolist()

print("\n" + "=" * 70)
print("PENALIZED REGRESSION (ELASTIC NET) ON KIC1 PER-SNP FEATURES — RSA")
print("=" * 70)
print(f"Design matrix shape: {X_ml.shape}")
print(f"Features (SNPs):     {len(snp_names)}")
print(f"Samples:             {X_ml.shape[0]}")

# ------------------------------------------------------------
# STEP 2 + 3: Elastic net with LOOCV over (alpha, l1_ratio)
# sklearn parameterization:
#   alpha       <-> lambda  (penalty strength)
#   l1_ratio    <-> alpha   (mixing: 1 = lasso, 0 = ridge)
# ------------------------------------------------------------
loo = LeaveOneOut()
l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]

enet_cv = ElasticNetCV(
    l1_ratio=l1_ratios,
    n_alphas=100,
    cv=loo,
    max_iter=20000,
    fit_intercept=True,
    random_state=0,
)

enet_cv.fit(X_ml, y_ml)

print("\n--- Selected hyperparameters (LOOCV) ---")
print(f"Optimal lambda (sklearn alpha): {enet_cv.alpha_:.6f}")
print(f"Optimal mixing l1_ratio:        {enet_cv.l1_ratio_:.2f}")

# ------------------------------------------------------------
# Refit at the optimal hyperparameters on full data
# ------------------------------------------------------------
enet_final = ElasticNet(
    alpha=enet_cv.alpha_,
    l1_ratio=enet_cv.l1_ratio_,
    max_iter=20000,
    fit_intercept=True,
)
enet_final.fit(X_ml, y_ml)

coef_table = pd.DataFrame({
    "SNP_ID": snp_names,
    "MB_weight": weights[snp_names].values,
    "Coefficient": enet_final.coef_,
    "Selected": enet_final.coef_ != 0,
})

print("\n--- Final coefficients (RSA ~ weighted SNPs) ---")
print(coef_table.to_string(index=False))
print(f"\nNumber of SNPs with non-zero coefficient: "
      f"{int(coef_table['Selected'].sum())} / {len(snp_names)}")

# ------------------------------------------------------------
# STEP 4: Bootstrap stability selection
# pi_i = fraction of bootstrap resamples where SNP i is selected
# ------------------------------------------------------------
B = 1000
rng_boot = np.random.default_rng(42)
n_samples = X_ml.shape[0]
selection_matrix = np.zeros((B, len(snp_names)), dtype=int)

for b in range(B):
    idx = rng_boot.choice(n_samples, size=n_samples, replace=True)
    X_b, y_b = X_ml[idx], y_ml[idx]
    try:
        model_b = ElasticNet(
            alpha=enet_cv.alpha_,
            l1_ratio=enet_cv.l1_ratio_,
            max_iter=20000,
            fit_intercept=True,
        )
        model_b.fit(X_b, y_b)
        selection_matrix[b] = (model_b.coef_ != 0).astype(int)
    except Exception:
        selection_matrix[b] = 0

stability = selection_matrix.mean(axis=0)

stability_table = pd.DataFrame({
    "SNP_ID": snp_names,
    "MB_weight": weights[snp_names].values,
    "Final_coef": enet_final.coef_,
    "Stability_pi": stability,
    "Robust_pi>=0.8": stability >= 0.8,
}).sort_values("Stability_pi", ascending=False)

print("\n--- Stability selection (B = 1000 bootstrap resamples) ---")
print(stability_table.to_string(index=False))

# ------------------------------------------------------------
# STEP 5: Permutation test on full-model R^2
# ------------------------------------------------------------
y_pred_obs = enet_final.predict(X_ml)
r2_obs = r2_score(y_ml, y_pred_obs)

n_perm = 1000
rng_perm = np.random.default_rng(7)
r2_null = np.zeros(n_perm)

for b in range(n_perm):
    y_perm = rng_perm.permutation(y_ml)
    model_p = ElasticNet(
        alpha=enet_cv.alpha_,
        l1_ratio=enet_cv.l1_ratio_,
        max_iter=20000,
        fit_intercept=True,
    )
    model_p.fit(X_ml, y_perm)
    r2_null[b] = r2_score(y_perm, model_p.predict(X_ml))

p_perm_r2 = (1 + np.sum(r2_null >= r2_obs)) / (1 + n_perm)

print("\n--- Permutation test of model R^2 (RSA) ---")
print(f"Observed in-sample R^2: {r2_obs:.4f}")
print(f"Null R^2 mean:          {r2_null.mean():.4f}")
print(f"Null R^2 95th pct:      {np.quantile(r2_null, 0.95):.4f}")
print(f"Permutation p-value:    {p_perm_r2:.4f}")

# ------------------------------------------------------------
# PLOT: Stability scores per SNP
# ------------------------------------------------------------
plt.figure(figsize=(9, 5))
order = np.argsort(stability)[::-1]
plt.bar(range(len(snp_names)),
        stability[order],
        color=["black" if stability[i] >= 0.8 else "grey" for i in order])
plt.axhline(0.8, linestyle="--", linewidth=1)
plt.xticks(range(len(snp_names)),
           [snp_names[i] for i in order],
           rotation=45, ha="right", fontsize=8)
plt.ylabel("Stability score π_i")
plt.ylim(0, 1.05)
plt.title("Bootstrap stability selection of KIC1 SNPs (Elastic Net, RSA, B=1000)")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# PLOT: Observed vs null R^2 distribution
# ------------------------------------------------------------
plt.figure(figsize=(8, 5))
plt.hist(r2_null, bins=40, color="grey", edgecolor="white")
plt.axvline(r2_obs, color="black", linewidth=2)
plt.xlabel("R² (in-sample)")
plt.ylabel("Frequency")
plt.title(f"Permutation null vs observed R² (RSA)\nObserved={r2_obs:.3f}, p={p_perm_r2:.3f}")
plt.tight_layout()
plt.show()

Genomewide association using RSA

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

from scipy.stats import ttest_ind, mannwhitneyu, ranksums
from statsmodels.stats.multitest import multipletests

# ============================================================
# FILE
# ============================================================
possible_files = [
    "ALL_BD_ISOLATE_VCF_TO_CSV - Sheet1.csv",
    "ALL_BD_ISOLATE_VCF_TO_CSV - Sheet1(2).csv",
    "/mnt/data/ALL_BD_ISOLATE_VCF_TO_CSV - Sheet1.csv",
    "/mnt/data/ALL_BD_ISOLATE_VCF_TO_CSV - Sheet1(2).csv"
]

vcf_csv = None
for f in possible_files:
    if os.path.exists(f):
        vcf_csv = f
        break

if vcf_csv is None:
    raise FileNotFoundError("VCF CSV file not found.")

print("Using file:", vcf_csv)

geno = pd.read_csv(vcf_csv, low_memory=False)
geno.columns = [str(c).strip() for c in geno.columns]

# ============================================================
# PARAMETERS
# ============================================================
MAX_DIST = 50_000
MIN_SNPS_PER_GENE = 2

MIN_ALT_COUNT = 2
MIN_REF_COUNT = 2
MIN_UNIQUE_SCORES = 3
MIN_SCORE_SD = 1e-6

# ============================================================
# RSA PHENOTYPE
# RSA >= 1 = resistant
# RSA < 1 = sensitive
# ============================================================
rsa = pd.DataFrame({
    "Sample": [
        "I-001-clone", "I-003-clone", "I-004", "I-011", "I-016",
        "I-018", "I-020", "I-029", "I-026", "I-033",
        "I-039", "I-041", "I-040", "I-027", "O-024",
        "I-008", "I-002", "O-014", "O-009", "O-032",
        "O-038", "I-019"
    ],
    "RSA": [
        0.6635725467, 2.007394647, 0, 0.6057562733, 4.634666667,
        0.7466666667, 1.76, 6.443333333, 0.6533333333, 1.889666667,
        4.891666667, 1.358666667, 0.7466666667, 0.5593333333, 1.73,
        1.535666667, 2.563333333, 2.126666667, 0.42, 0.783,
        1.371666667, 0.4633333333
    ]
})

rsa["Sample"] = rsa["Sample"].astype(str).str.strip()
rsa["RSA"] = pd.to_numeric(rsa["RSA"], errors="coerce")
rsa = rsa.dropna()

rsa["Binary"] = (rsa["RSA"] >= 1).astype(int)
rsa["Group"] = np.where(rsa["RSA"] >= 1, "Resistant_RSA", "Sensitive_RSA")
rsa = rsa.set_index("Sample")

print("\nRSA group counts:")
print(rsa["Group"].value_counts())

# ============================================================
# SAMPLE MATCHING
# ============================================================
if "FORMAT" not in geno.columns:
    raise ValueError("FORMAT column not found.")

format_idx = list(geno.columns).index("FORMAT")
vcf_samples = [str(x).strip() for x in geno.columns[format_idx + 1:]]

common = sorted(set(vcf_samples).intersection(set(rsa.index)))

print("\nSamples in VCF:", len(vcf_samples))
print("Samples with RSA:", rsa.shape[0])
print("Matched samples:", len(common))
print(common)

if len(common) == 0:
    raise ValueError("No matching samples between RSA table and VCF CSV.")

rsa_data = rsa.loc[common].copy()

# ============================================================
# REQUIRED COLUMNS
# ============================================================
chrom_col = "#CHROM" if "#CHROM" in geno.columns else "CHROM"

for c in [chrom_col, "POS", "REF", "ALT", "INFO", "FORMAT"]:
    if c not in geno.columns:
        raise ValueError(f"Missing required column: {c}")

geno["POS"] = pd.to_numeric(geno["POS"], errors="coerce")
geno = geno.dropna(subset=["POS"]).copy()
geno["POS"] = geno["POS"].astype(int)

geno["SNP_ID"] = geno.apply(
    lambda r: f"{r[chrom_col]}:{int(r['POS'])}:{r['REF']}>{r['ALT']}",
    axis=1
)

# ============================================================
# ALT MATRIX
# ============================================================
def alt_present(x):
    if pd.isna(x):
        return 0.0

    gt = str(x).split(":")[0]

    if gt in ["./.", ".|.", ".", "nan", "None"]:
        return 0.0

    alleles = re.split(r"[\/|]", gt)
    return 1.0 if "1" in alleles else 0.0

A = pd.DataFrame(
    {s: geno[s].map(alt_present).values for s in common},
    index=geno["SNP_ID"]
).T

print("\nALT matrix shape:", A.shape)

# ============================================================
# PARSE ANN
# Drop synonymous variants
# Keep intergenic SNPs, collapse to closest candidate gene within 50 kb
# ============================================================
pf_gene_pattern = re.compile(r"PF3D7_\d{7}")

def clean_common_name(x):
    x = str(x).strip()

    if x in ["", ".", "nan", "NaN", "None", "null", "NULL"]:
        return ""

    if pf_gene_pattern.fullmatch(x):
        return ""

    if "_circ" in x:
        return ""

    return x

def is_synonymous_effect(effect):
    return "synonymous_variant" in str(effect)

gene_body_records = []
intergenic_raw_records = []
dropped_synonymous = 0

for _, row in geno.iterrows():

    info = str(row["INFO"])
    snp_id = row["SNP_ID"]
    chrom = row[chrom_col]
    pos = row["POS"]

    ann_field = None

    for field in info.split(";"):
        if field.startswith("ANN="):
            ann_field = field[4:]
            break

    if ann_field is None:
        continue

    for ann_record in ann_field.split(","):

        parts = ann_record.split("|")

        if len(parts) < 7:
            continue

        effect = parts[1]
        impact = parts[2]
        gene_name_field = parts[3]
        gene_id_field = parts[4]
        feature_type = parts[5]
        feature_id = parts[6]

        if is_synonymous_effect(effect):
            dropped_synonymous += 1
            continue

        text = "|".join([gene_id_field, feature_id, gene_name_field, ann_record])
        gene_ids = sorted(set(pf_gene_pattern.findall(text)))

        if len(gene_ids) == 0:
            continue

        if "intergenic_region" in effect:

            raw_common_names = str(gene_name_field).split("-")
            clean_names = [clean_common_name(x) for x in raw_common_names]
            clean_names = [x for x in clean_names if x != ""]

            intergenic_raw_records.append({
                "SNP_ID": snp_id,
                "CHROM": chrom,
                "POS": pos,
                "REF": row["REF"],
                "ALT": row["ALT"],
                "Candidate_Gene_IDs": gene_ids,
                "Candidate_common_names": clean_names,
                "Effect": effect,
                "Impact": impact,
                "Feature_type": feature_type,
                "Feature_ID": feature_id
            })

        else:

            common_name = clean_common_name(gene_name_field)

            for gene_id in gene_ids:
                gene_body_records.append({
                    "SNP_ID": snp_id,
                    "Gene_ID": gene_id,
                    "Common_name": common_name,
                    "CHROM": chrom,
                    "POS": pos,
                    "REF": row["REF"],
                    "ALT": row["ALT"],
                    "Effect": effect,
                    "Impact": impact,
                    "Feature_type": feature_type,
                    "Feature_ID": feature_id,
                    "Is_intergenic": False,
                    "Distance_to_gene_bp": 0,
                    "Assignment_rule": "gene_body_or_transcript_non_synonymous"
                })

gene_body_df = pd.DataFrame(gene_body_records)
intergenic_raw_df = pd.DataFrame(intergenic_raw_records)

print("\nDropped synonymous ANN records:", dropped_synonymous)
print("Gene-body/transcript non-synonymous assignments:", gene_body_df.shape[0])
print("Raw intergenic SNP records:", intergenic_raw_df.shape[0])

if gene_body_df.empty:
    raise ValueError("No non-synonymous gene-body/transcript annotations found.")

# ============================================================
# BUILD OBSERVED GENE INTERVALS FROM NON-SYNONYMOUS GENE-BODY SNPs
# ============================================================
gene_intervals = gene_body_df.groupby("Gene_ID").agg(
    CHROM=("CHROM", "first"),
    Gene_start_observed=("POS", "min"),
    Gene_end_observed=("POS", "max"),
    Gene_mid_observed=("POS", "median")
).reset_index()

gene_interval_dict = gene_intervals.set_index("Gene_ID").to_dict("index")

print("\nObserved gene intervals built for genes:", gene_intervals.shape[0])

# ============================================================
# COLLAPSE INTERGENIC SNPs TO CLOSEST CANDIDATE GENE
# ============================================================
intergenic_assigned_records = []

for _, row in intergenic_raw_df.iterrows():

    snp_pos = row["POS"]
    snp_chrom = row["CHROM"]

    candidate_rows = []

    for gene_id in row["Candidate_Gene_IDs"]:

        if gene_id not in gene_interval_dict:
            continue

        g = gene_interval_dict[gene_id]

        if g["CHROM"] != snp_chrom:
            continue

        gene_start = int(g["Gene_start_observed"])
        gene_end = int(g["Gene_end_observed"])

        if snp_pos < gene_start:
            dist = gene_start - snp_pos
        elif snp_pos > gene_end:
            dist = snp_pos - gene_end
        else:
            dist = 0

        candidate_rows.append({
            "Gene_ID": gene_id,
            "Distance_to_gene_bp": dist,
            "Gene_start_observed": gene_start,
            "Gene_end_observed": gene_end
        })

    if len(candidate_rows) == 0:
        continue

    candidate_df = pd.DataFrame(candidate_rows)

    min_dist = candidate_df["Distance_to_gene_bp"].min()

    if min_dist > MAX_DIST:
        continue

    closest_genes = candidate_df[
        candidate_df["Distance_to_gene_bp"] == min_dist
    ].copy()

    for _, closest in closest_genes.iterrows():

        gene_id = closest["Gene_ID"]

        common_name = ""

        if len(row["Candidate_common_names"]) == len(row["Candidate_Gene_IDs"]):
            gene_to_name = dict(zip(row["Candidate_Gene_IDs"], row["Candidate_common_names"]))
            common_name = gene_to_name.get(gene_id, "")

        if common_name == "":
            common_name = ";".join(row["Candidate_common_names"])

        intergenic_assigned_records.append({
            "SNP_ID": row["SNP_ID"],
            "Gene_ID": gene_id,
            "Common_name": common_name,
            "CHROM": row["CHROM"],
            "POS": row["POS"],
            "REF": row["REF"],
            "ALT": row["ALT"],
            "Effect": row["Effect"],
            "Impact": row["Impact"],
            "Feature_type": row["Feature_type"],
            "Feature_ID": row["Feature_ID"],
            "Is_intergenic": True,
            "Distance_to_gene_bp": float(min_dist),
            "Assignment_rule": "closest_intergenic_candidate_within_50kb"
        })

intergenic_df = pd.DataFrame(intergenic_assigned_records)

print("\nIntergenic SNPs assigned after closest-gene 50 kb filter:", intergenic_df.shape[0])

if not intergenic_df.empty:
    print("\nIntergenic distance summary:")
    print(intergenic_df["Distance_to_gene_bp"].describe())

# ============================================================
# FINAL ASSIGNMENT TABLE
# ============================================================
assign = pd.concat(
    [gene_body_df, intergenic_df],
    ignore_index=True
)

assign = assign.drop_duplicates(
    subset=["SNP_ID", "Gene_ID", "Effect", "Is_intergenic"]
).copy()

assign = assign[
    ~assign["Effect"].astype(str).str.contains("synonymous_variant", na=False)
].copy()

print("\nTotal SNP-to-gene assignments after dropping synonymous:", assign.shape[0])
print("Unique SNPs assigned:", assign["SNP_ID"].nunique())
print("Unique genes assigned:", assign["Gene_ID"].nunique())

print("\nAssignment type:")
print(assign["Is_intergenic"].value_counts())

print("\nTop effects after dropping synonymous:")
print(assign["Effect"].value_counts().head(30))

assign.to_csv("SNP_to_gene_assignments_no_synonymous_closest_intergenic_50kb.csv", index=False)

# ============================================================
# COMMON GENE NAMES
# ============================================================
def collapse_names(x):
    vals = []

    for item in x.dropna().astype(str):
        for z in item.split(";"):
            z = clean_common_name(z)
            if z != "":
                vals.append(z)

    return ";".join(sorted(set(vals)))

gene_names = assign.groupby("Gene_ID")["Common_name"].apply(
    collapse_names
).reset_index()

# ============================================================
# GENE POSITION TABLE
# ============================================================
gene_position = assign.groupby("Gene_ID").agg(
    CHROM=("CHROM", "first"),
    Gene_mid_POS=("POS", "median"),
    Min_POS=("POS", "min"),
    Max_POS=("POS", "max")
).reset_index()

# ============================================================
# GENE SCORE ASSOCIATION WITH VARIATION FILTERS
# ============================================================
results = []

for gene_id, gdf in assign.groupby("Gene_ID"):

    snps_all = sorted(set(gdf["SNP_ID"]).intersection(A.columns))

    if len(snps_all) == 0:
        continue

    A_gene = A[snps_all].copy()

    alt_counts = A_gene.sum(axis=0)
    n_samples = A_gene.shape[0]
    ref_counts = n_samples - alt_counts

    keep_snps = alt_counts[
        (alt_counts >= MIN_ALT_COUNT) &
        (ref_counts >= MIN_REF_COUNT)
    ].index.tolist()

    if len(keep_snps) < MIN_SNPS_PER_GENE:
        continue

    A_gene = A_gene[keep_snps].copy()

    maf = A_gene.mean(axis=0)

    variable_snps = maf[(maf > 0) & (maf < 1)].index.tolist()

    if len(variable_snps) < MIN_SNPS_PER_GENE:
        continue

    A_gene = A_gene[variable_snps].copy()
    maf = A_gene.mean(axis=0)

    weights = 1 / np.sqrt(maf * (1 - maf))
    weights = weights.replace([np.inf, -np.inf], 0).fillna(0)

    gene_score = ((A_gene * weights) ** 2).sum(axis=1)

    df = pd.DataFrame({
        "Score": gene_score,
        "RSA": rsa_data.loc[gene_score.index, "RSA"],
        "Binary": rsa_data.loc[gene_score.index, "Binary"],
        "Group": rsa_data.loc[gene_score.index, "Group"]
    }).dropna()

    if df["Score"].nunique() < MIN_UNIQUE_SCORES:
        continue

    if df["Score"].std() <= MIN_SCORE_SD:
        continue

    sensitive = df[df["Binary"] == 0]["Score"]
    resistant = df[df["Binary"] == 1]["Score"]

    if len(sensitive) < 2 or len(resistant) < 2:
        continue

    if sensitive.nunique() < 2 and resistant.nunique() < 2:
        continue

    try:
        X = sm.add_constant(df["Score"])
        fit = sm.OLS(df["RSA"], X).fit()
        ols_beta = fit.params.iloc[1]
        ols_p = fit.pvalues.iloc[1]
        ols_r2 = fit.rsquared
    except Exception:
        ols_beta = np.nan
        ols_p = np.nan
        ols_r2 = np.nan

    try:
        welch_p = ttest_ind(sensitive, resistant, equal_var=False).pvalue
    except Exception:
        welch_p = np.nan

    try:
        mw_p = mannwhitneyu(sensitive, resistant, alternative="two-sided").pvalue
    except Exception:
        mw_p = np.nan

    try:
        rank_p = ranksums(sensitive, resistant).pvalue
    except Exception:
        rank_p = np.nan

    try:
        logit = sm.Logit(df["Binary"], sm.add_constant(df["Score"])).fit(disp=0)
        logit_beta = logit.params.iloc[1]
        logit_p = logit.pvalues.iloc[1]
        logit_or = np.exp(logit_beta)
    except Exception:
        logit_beta = np.nan
        logit_p = np.nan
        logit_or = np.nan

    gdf_used = gdf[gdf["SNP_ID"].isin(variable_snps)].copy()

    n_intergenic = int(gdf_used["Is_intergenic"].sum())
    n_gene_body = int((~gdf_used["Is_intergenic"]).sum())

    intergenic_dist = gdf_used.loc[
        gdf_used["Is_intergenic"],
        "Distance_to_gene_bp"
    ]

    max_dist = intergenic_dist.max() if len(intergenic_dist) > 0 else 0
    mean_dist = intergenic_dist.mean() if len(intergenic_dist) > 0 else 0

    results.append({
        "Gene_ID": gene_id,
        "CHROM": gdf["CHROM"].iloc[0],
        "Gene_mid_POS": gdf["POS"].median(),
        "Min_POS": gdf["POS"].min(),
        "Max_POS": gdf["POS"].max(),
        "N_SNPs_used": len(variable_snps),
        "N_intergenic_SNP_assignments_50kb": n_intergenic,
        "N_gene_body_or_transcript_SNP_assignments": n_gene_body,
        "Max_intergenic_distance_bp": max_dist,
        "Mean_intergenic_distance_bp": mean_dist,
        "SNPs_used": ";".join(variable_snps),
        "Effects_used": ";".join(sorted(gdf_used["Effect"].dropna().unique())),
        "Mean_score_sensitive_RSA_lt_1": sensitive.mean(),
        "Mean_score_resistant_RSA_ge_1": resistant.mean(),
        "Difference_resistant_minus_sensitive": resistant.mean() - sensitive.mean(),
        "OLS_beta_RSA": ols_beta,
        "OLS_p_RSA": ols_p,
        "OLS_R2_RSA": ols_r2,
        "Welch_p": welch_p,
        "MannWhitney_p": mw_p,
        "Wilcoxon_rank_sum_p": rank_p,
        "Logistic_beta": logit_beta,
        "Logistic_p": logit_p,
        "Logistic_OR": logit_or
    })

res = pd.DataFrame(results)

print("\nNumber of tested genes:", res.shape[0])
print("Result columns:", res.columns.tolist())

if res.empty:
    raise ValueError("No genes passed filters.")

# ============================================================
# ADD COMMON NAME
# ============================================================
res = res.merge(gene_names, on="Gene_ID", how="left")

# ============================================================
# FDR
# ============================================================
for pcol in ["OLS_p_RSA", "Welch_p", "MannWhitney_p", "Logistic_p"]:

    res[pcol + "_FDR"] = np.nan

    valid = res[pcol].notna() & (res[pcol] > 0) & (res[pcol] <= 1)

    if valid.sum() > 0:
        res.loc[valid, pcol + "_FDR"] = multipletests(
            res.loc[valid, pcol],
            method="fdr_bh"
        )[1]

res = res.sort_values("OLS_p_RSA", na_position="last").reset_index(drop=True)

# ============================================================
# FINAL TABLE
# ============================================================
final_cols = [
    "Gene_ID",
    "Common_name",
    "CHROM",
    "Gene_mid_POS",
    "Min_POS",
    "Max_POS",
    "N_SNPs_used",
    "N_intergenic_SNP_assignments_50kb",
    "N_gene_body_or_transcript_SNP_assignments",
    "Max_intergenic_distance_bp",
    "Mean_intergenic_distance_bp",
    "Effects_used",
    "Mean_score_sensitive_RSA_lt_1",
    "Mean_score_resistant_RSA_ge_1",
    "Difference_resistant_minus_sensitive",
    "OLS_beta_RSA",
    "OLS_p_RSA",
    "OLS_p_RSA_FDR",
    "OLS_R2_RSA",
    "Welch_p",
    "Welch_p_FDR",
    "MannWhitney_p",
    "MannWhitney_p_FDR",
    "Logistic_p",
    "Logistic_p_FDR",
    "Logistic_OR",
    "SNPs_used"
]

final_table = res[final_cols].copy()

out_csv = "Genomewide_RSA_gene_score_NO_SYNONYMOUS_filtered_FIXED_MANHATTAN.csv"
final_table.to_csv(out_csv, index=False)

print("\nTop 30 genome-wide gene results:")
print(final_table.head(30).to_string(index=False))

print("\nSaved final table:")
print(out_csv)

# ============================================================
# QQ PLOT
# ============================================================
qq = res.dropna(subset=["OLS_p_RSA"]).copy()
qq = qq[(qq["OLS_p_RSA"] > 0) & (qq["OLS_p_RSA"] <= 1)].copy()

pvals = np.sort(qq["OLS_p_RSA"].values)

observed = -np.log10(pvals)
expected = -np.log10(np.arange(1, len(pvals) + 1) / (len(pvals) + 1))

plt.figure(figsize=(6, 6))
plt.scatter(expected, observed, s=25)

max_val = max(expected.max(), observed.max())
plt.plot([0, max_val], [0, max_val], linestyle="--", linewidth=1)

plt.xlabel("Expected -log10(p)")
plt.ylabel("Observed -log10(p)")
plt.title("QQ plot: RSA gene-score OLS p-values, no synonymous")

plt.tight_layout()
plt.show()

# ============================================================
# MANHATTAN PLOT - FIXED
# Uses CHROM, not Gene_ID, to infer chromosome number
# ============================================================
def chrom_number(x):
    x = str(x)
    m = re.search(r"Pf3D7_(\d+)_v3", x)
    return int(m.group(1)) if m else np.nan

man = res.dropna(subset=["OLS_p_RSA", "CHROM", "Gene_mid_POS"]).copy()
man = man[(man["OLS_p_RSA"] > 0) & (man["OLS_p_RSA"] <= 1)].copy()

man["CHR_NUM"] = man["CHROM"].apply(chrom_number)

print("\nRows before Manhattan chromosome filtering:", man.shape[0])
print(man[["Gene_ID", "CHROM", "CHR_NUM", "Gene_mid_POS", "OLS_p_RSA"]].head(20))

man = man.dropna(subset=["CHR_NUM"]).copy()

if man.empty:
    raise ValueError("No valid rows for Manhattan plot after CHROM parsing.")

man["CHR_NUM"] = man["CHR_NUM"].astype(int)
man["Gene_mid_POS"] = pd.to_numeric(man["Gene_mid_POS"], errors="coerce")
man = man.dropna(subset=["Gene_mid_POS"]).copy()

man = man.sort_values(["CHR_NUM", "Gene_mid_POS"]).copy()

chr_centers = {}
offset = 0
cum_positions = []

for chrom in sorted(man["CHR_NUM"].unique()):

    sub = man[man["CHR_NUM"] == chrom].copy()

    x = sub["Gene_mid_POS"] + offset
    cum_positions.extend(x.tolist())

    chr_centers[chrom] = x.mean()

    offset = x.max() + 500000

man["BP_cum"] = cum_positions
man["minus_log10_p"] = -np.log10(man["OLS_p_RSA"])

plt.figure(figsize=(14, 6))

for chrom in sorted(man["CHR_NUM"].unique()):
    sub = man[man["CHR_NUM"] == chrom]

    plt.scatter(
        sub["BP_cum"],
        sub["minus_log10_p"],
        s=30
    )

bonf = 0.05 / man.shape[0]

plt.axhline(-np.log10(bonf), linestyle="--", linewidth=1)
plt.axhline(-np.log10(0.05), linestyle=":", linewidth=1)

plt.xticks(
    [chr_centers[c] for c in sorted(chr_centers)],
    [str(c) for c in sorted(chr_centers)]
)

plt.xlabel("Chromosome")
plt.ylabel("-log10(OLS p-value)")
plt.title("Genome-wide Manhattan plot: RSA gene score, no synonymous, filtered")

plt.tight_layout()
plt.show()

# ============================================================
# LABELED MANHATTAN
# ============================================================
top_n = 15
top_hits = man.nsmallest(top_n, "OLS_p_RSA").copy()

plt.figure(figsize=(14, 6))

for chrom in sorted(man["CHR_NUM"].unique()):
    sub = man[man["CHR_NUM"] == chrom]

    plt.scatter(
        sub["BP_cum"],
        sub["minus_log10_p"],
        s=30
    )

plt.axhline(-np.log10(bonf), linestyle="--", linewidth=1)
plt.axhline(-np.log10(0.05), linestyle=":", linewidth=1)

for _, row in top_hits.iterrows():

    common_name = str(row["Common_name"]).strip()

    if common_name in ["", "nan", "None"]:
        label = row["Gene_ID"]
    else:
        label = common_name.split(";")[0]

    plt.text(
        row["BP_cum"],
        row["minus_log10_p"] + 0.08,
        label,
        fontsize=8,
        rotation=75,
        ha="left"
    )

plt.xticks(
    [chr_centers[c] for c in sorted(chr_centers)],
    [str(c) for c in sorted(chr_centers)]
)

plt.xlabel("Chromosome")
plt.ylabel("-log10(OLS p-value)")
plt.title("Genome-wide Manhattan plot with top labeled genes")

plt.tight_layout()
plt.show()

# ============================================================
# SANITY CHECK
# ============================================================
print("\nExamples of closest intergenic SNP assignments within 50 kb:")
if intergenic_df.empty:
    print("No intergenic SNPs were assigned within 50 kb.")
else:
    print(
        intergenic_df[
            ["SNP_ID", "Gene_ID", "Common_name", "Distance_to_gene_bp", "Effect"]
        ]
        .drop_duplicates()
        .head(50)
        .to_string(index=False)
    )

print("\nMaximum intergenic distance included:")
if intergenic_df.empty:
    print("None")
else:
    print(intergenic_df["Distance_to_gene_bp"].max())

print("\nCheck that synonymous variants were removed:")
print(assign["Effect"].astype(str).str.contains("synonymous_variant", na=False).value_counts())

Genomewide association using PC50

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

from scipy.stats import ttest_ind, mannwhitneyu, ranksums
from statsmodels.stats.multitest import multipletests

# ============================================================
# FILE
# ============================================================
possible_files = [
    "ALL_BD_ISOLATE_VCF_TO_CSV - Sheet1.csv",
    "ALL_BD_ISOLATE_VCF_TO_CSV - Sheet1(2).csv",
    "/mnt/data/ALL_BD_ISOLATE_VCF_TO_CSV - Sheet1.csv",
    "/mnt/data/ALL_BD_ISOLATE_VCF_TO_CSV - Sheet1(2).csv"
]

vcf_csv = None
for f in possible_files:
    if os.path.exists(f):
        vcf_csv = f
        break

if vcf_csv is None:
    raise FileNotFoundError("VCF CSV file not found.")

print("Using file:", vcf_csv)

geno = pd.read_csv(vcf_csv, low_memory=False)
geno.columns = [str(c).strip() for c in geno.columns]

# ============================================================
# PARAMETERS
# ============================================================
MAX_DIST = 50_000
MIN_SNPS_PER_GENE = 2

MIN_ALT_COUNT = 2
MIN_REF_COUNT = 2
MIN_UNIQUE_SCORES = 3
MIN_SCORE_SD = 1e-6

# ============================================================
# PC50 PHENOTYPE
# PC50 >= 5 = delayed/high PC50
# PC50 < 5 = low PC50
# ============================================================
pc50 = pd.DataFrame({
    "Sample": [
        "I-001-clone", "I-002", "I-003-clone", "I-004", "I-005",
        "I-008", "I-010", "I-011", "I-013", "I-016",
        "I-017", "I-018", "I-019", "I-020", "I-021",
        "I-025", "I-026", "I-027", "I-029", "I-030",
        "I-031", "I-033", "I-035", "I-036", "I-039",
        "I-040-", "I-041"
    ],
    "PC50": [
        0.015, 3.56, 5.967, 1.6, 3.82,
        2.71, 2.01, 5.35, 7.98, 10.25,
        4.51, 3.42, 2.02, 7.66, 2.83,
        3.39, 3.25, 3.23, 7.68, 8.19,
        3.41, 8.48, 1.25, 9.42, 13.14,
        3.14, 4.82
    ]
})

pc50["Sample"] = pc50["Sample"].astype(str).str.strip()
pc50["PC50"] = pd.to_numeric(pc50["PC50"], errors="coerce")
pc50 = pc50.dropna()

pc50["Binary"] = (pc50["PC50"] >= 5).astype(int)
pc50["Group"] = np.where(pc50["PC50"] >= 5, "Delayed_PC50", "Low_PC50")
pc50 = pc50.set_index("Sample")

print("\nPC50 group counts:")
print(pc50["Group"].value_counts())

# ============================================================
# SAMPLE MATCHING
# ============================================================
if "FORMAT" not in geno.columns:
    raise ValueError("FORMAT column not found.")

format_idx = list(geno.columns).index("FORMAT")
vcf_samples = [str(x).strip() for x in geno.columns[format_idx + 1:]]

common = sorted(set(vcf_samples).intersection(set(pc50.index)))

print("\nSamples in VCF:", len(vcf_samples))
print("Samples with PC50:", pc50.shape[0])
print("Matched samples:", len(common))
print(common)

if len(common) == 0:
    raise ValueError("No matching samples between PC50 table and VCF CSV.")

pc50_data = pc50.loc[common].copy()

# ============================================================
# REQUIRED COLUMNS
# ============================================================
chrom_col = "#CHROM" if "#CHROM" in geno.columns else "CHROM"

for c in [chrom_col, "POS", "REF", "ALT", "INFO", "FORMAT"]:
    if c not in geno.columns:
        raise ValueError(f"Missing required column: {c}")

geno["POS"] = pd.to_numeric(geno["POS"], errors="coerce")
geno = geno.dropna(subset=["POS"]).copy()
geno["POS"] = geno["POS"].astype(int)

geno["SNP_ID"] = geno.apply(
    lambda r: f"{r[chrom_col]}:{int(r['POS'])}:{r['REF']}>{r['ALT']}",
    axis=1
)

# ============================================================
# ALT MATRIX
# ============================================================
def alt_present(x):
    if pd.isna(x):
        return 0.0

    gt = str(x).split(":")[0]

    if gt in ["./.", ".|.", ".", "nan", "None"]:
        return 0.0

    alleles = re.split(r"[\/|]", gt)
    return 1.0 if "1" in alleles else 0.0

A = pd.DataFrame(
    {s: geno[s].map(alt_present).values for s in common},
    index=geno["SNP_ID"]
).T

print("\nALT matrix shape:", A.shape)

# ============================================================
# PARSE ANN
# Drop synonymous variants
# Keep intergenic SNPs, collapse to closest candidate gene within 50 kb
# ============================================================
pf_gene_pattern = re.compile(r"PF3D7_\d{7}")

def clean_common_name(x):
    x = str(x).strip()

    if x in ["", ".", "nan", "NaN", "None", "null", "NULL"]:
        return ""

    if pf_gene_pattern.fullmatch(x):
        return ""

    if "_circ" in x:
        return ""

    return x

def is_synonymous_effect(effect):
    return "synonymous_variant" in str(effect)

gene_body_records = []
intergenic_raw_records = []
dropped_synonymous = 0

for _, row in geno.iterrows():

    info = str(row["INFO"])
    snp_id = row["SNP_ID"]
    chrom = row[chrom_col]
    pos = row["POS"]

    ann_field = None

    for field in info.split(";"):
        if field.startswith("ANN="):
            ann_field = field[4:]
            break

    if ann_field is None:
        continue

    for ann_record in ann_field.split(","):

        parts = ann_record.split("|")

        if len(parts) < 7:
            continue

        effect = parts[1]
        impact = parts[2]
        gene_name_field = parts[3]
        gene_id_field = parts[4]
        feature_type = parts[5]
        feature_id = parts[6]

        if is_synonymous_effect(effect):
            dropped_synonymous += 1
            continue

        text = "|".join([gene_id_field, feature_id, gene_name_field, ann_record])
        gene_ids = sorted(set(pf_gene_pattern.findall(text)))

        if len(gene_ids) == 0:
            continue

        if "intergenic_region" in effect:

            raw_common_names = str(gene_name_field).split("-")
            clean_names = [clean_common_name(x) for x in raw_common_names]
            clean_names = [x for x in clean_names if x != ""]

            intergenic_raw_records.append({
                "SNP_ID": snp_id,
                "CHROM": chrom,
                "POS": pos,
                "REF": row["REF"],
                "ALT": row["ALT"],
                "Candidate_Gene_IDs": gene_ids,
                "Candidate_common_names": clean_names,
                "Effect": effect,
                "Impact": impact,
                "Feature_type": feature_type,
                "Feature_ID": feature_id
            })

        else:

            common_name = clean_common_name(gene_name_field)

            for gene_id in gene_ids:
                gene_body_records.append({
                    "SNP_ID": snp_id,
                    "Gene_ID": gene_id,
                    "Common_name": common_name,
                    "CHROM": chrom,
                    "POS": pos,
                    "REF": row["REF"],
                    "ALT": row["ALT"],
                    "Effect": effect,
                    "Impact": impact,
                    "Feature_type": feature_type,
                    "Feature_ID": feature_id,
                    "Is_intergenic": False,
                    "Distance_to_gene_bp": 0,
                    "Assignment_rule": "gene_body_or_transcript_non_synonymous"
                })

gene_body_df = pd.DataFrame(gene_body_records)
intergenic_raw_df = pd.DataFrame(intergenic_raw_records)

print("\nDropped synonymous ANN records:", dropped_synonymous)
print("Gene-body/transcript non-synonymous assignments:", gene_body_df.shape[0])
print("Raw intergenic SNP records:", intergenic_raw_df.shape[0])

if gene_body_df.empty:
    raise ValueError("No non-synonymous gene-body/transcript annotations found.")

# ============================================================
# BUILD OBSERVED GENE INTERVALS FROM NON-SYNONYMOUS GENE-BODY SNPs
# ============================================================
gene_intervals = gene_body_df.groupby("Gene_ID").agg(
    CHROM=("CHROM", "first"),
    Gene_start_observed=("POS", "min"),
    Gene_end_observed=("POS", "max"),
    Gene_mid_observed=("POS", "median")
).reset_index()

gene_interval_dict = gene_intervals.set_index("Gene_ID").to_dict("index")

print("\nObserved gene intervals built for genes:", gene_intervals.shape[0])

# ============================================================
# COLLAPSE INTERGENIC SNPs TO CLOSEST CANDIDATE GENE
# ============================================================
intergenic_assigned_records = []

for _, row in intergenic_raw_df.iterrows():

    snp_pos = row["POS"]
    snp_chrom = row["CHROM"]

    candidate_rows = []

    for gene_id in row["Candidate_Gene_IDs"]:

        if gene_id not in gene_interval_dict:
            continue

        g = gene_interval_dict[gene_id]

        if g["CHROM"] != snp_chrom:
            continue

        gene_start = int(g["Gene_start_observed"])
        gene_end = int(g["Gene_end_observed"])

        if snp_pos < gene_start:
            dist = gene_start - snp_pos
        elif snp_pos > gene_end:
            dist = snp_pos - gene_end
        else:
            dist = 0

        candidate_rows.append({
            "Gene_ID": gene_id,
            "Distance_to_gene_bp": dist,
            "Gene_start_observed": gene_start,
            "Gene_end_observed": gene_end
        })

    if len(candidate_rows) == 0:
        continue

    candidate_df = pd.DataFrame(candidate_rows)

    min_dist = candidate_df["Distance_to_gene_bp"].min()

    if min_dist > MAX_DIST:
        continue

    closest_genes = candidate_df[
        candidate_df["Distance_to_gene_bp"] == min_dist
    ].copy()

    for _, closest in closest_genes.iterrows():

        gene_id = closest["Gene_ID"]

        common_name = ""

        if len(row["Candidate_common_names"]) == len(row["Candidate_Gene_IDs"]):
            gene_to_name = dict(zip(row["Candidate_Gene_IDs"], row["Candidate_common_names"]))
            common_name = gene_to_name.get(gene_id, "")

        if common_name == "":
            common_name = ";".join(row["Candidate_common_names"])

        intergenic_assigned_records.append({
            "SNP_ID": row["SNP_ID"],
            "Gene_ID": gene_id,
            "Common_name": common_name,
            "CHROM": row["CHROM"],
            "POS": row["POS"],
            "REF": row["REF"],
            "ALT": row["ALT"],
            "Effect": row["Effect"],
            "Impact": row["Impact"],
            "Feature_type": row["Feature_type"],
            "Feature_ID": row["Feature_ID"],
            "Is_intergenic": True,
            "Distance_to_gene_bp": float(min_dist),
            "Assignment_rule": "closest_intergenic_candidate_within_50kb"
        })

intergenic_df = pd.DataFrame(intergenic_assigned_records)

print("\nIntergenic SNPs assigned after closest-gene 50 kb filter:", intergenic_df.shape[0])

if not intergenic_df.empty:
    print("\nIntergenic distance summary:")
    print(intergenic_df["Distance_to_gene_bp"].describe())

# ============================================================
# FINAL ASSIGNMENT TABLE
# ============================================================
assign = pd.concat(
    [gene_body_df, intergenic_df],
    ignore_index=True
)

assign = assign.drop_duplicates(
    subset=["SNP_ID", "Gene_ID", "Effect", "Is_intergenic"]
).copy()

assign = assign[
    ~assign["Effect"].astype(str).str.contains("synonymous_variant", na=False)
].copy()

print("\nTotal SNP-to-gene assignments after dropping synonymous:", assign.shape[0])
print("Unique SNPs assigned:", assign["SNP_ID"].nunique())
print("Unique genes assigned:", assign["Gene_ID"].nunique())

print("\nAssignment type:")
print(assign["Is_intergenic"].value_counts())

print("\nTop effects after dropping synonymous:")
print(assign["Effect"].value_counts().head(30))

assign.to_csv("SNP_to_gene_assignments_PC50_no_synonymous_closest_intergenic_50kb.csv", index=False)

# ============================================================
# COMMON GENE NAMES
# ============================================================
def collapse_names(x):
    vals = []

    for item in x.dropna().astype(str):
        for z in item.split(";"):
            z = clean_common_name(z)
            if z != "":
                vals.append(z)

    return ";".join(sorted(set(vals)))

gene_names = assign.groupby("Gene_ID")["Common_name"].apply(
    collapse_names
).reset_index()

# ============================================================
# GENE SCORE ASSOCIATION WITH VARIATION FILTERS
# ============================================================
results = []

for gene_id, gdf in assign.groupby("Gene_ID"):

    snps_all = sorted(set(gdf["SNP_ID"]).intersection(A.columns))

    if len(snps_all) == 0:
        continue

    A_gene = A[snps_all].copy()

    alt_counts = A_gene.sum(axis=0)
    n_samples = A_gene.shape[0]
    ref_counts = n_samples - alt_counts

    keep_snps = alt_counts[
        (alt_counts >= MIN_ALT_COUNT) &
        (ref_counts >= MIN_REF_COUNT)
    ].index.tolist()

    if len(keep_snps) < MIN_SNPS_PER_GENE:
        continue

    A_gene = A_gene[keep_snps].copy()

    maf = A_gene.mean(axis=0)

    variable_snps = maf[(maf > 0) & (maf < 1)].index.tolist()

    if len(variable_snps) < MIN_SNPS_PER_GENE:
        continue

    A_gene = A_gene[variable_snps].copy()
    maf = A_gene.mean(axis=0)

    weights = 1 / np.sqrt(maf * (1 - maf))
    weights = weights.replace([np.inf, -np.inf], 0).fillna(0)

    gene_score = ((A_gene * weights) ** 2).sum(axis=1)

    df = pd.DataFrame({
        "Score": gene_score,
        "PC50": pc50_data.loc[gene_score.index, "PC50"],
        "Binary": pc50_data.loc[gene_score.index, "Binary"],
        "Group": pc50_data.loc[gene_score.index, "Group"]
    }).dropna()

    if df["Score"].nunique() < MIN_UNIQUE_SCORES:
        continue

    if df["Score"].std() <= MIN_SCORE_SD:
        continue

    low = df[df["Binary"] == 0]["Score"]
    delayed = df[df["Binary"] == 1]["Score"]

    if len(low) < 2 or len(delayed) < 2:
        continue

    if low.nunique() < 2 and delayed.nunique() < 2:
        continue

    try:
        X = sm.add_constant(df["Score"])
        fit = sm.OLS(df["PC50"], X).fit()
        ols_beta = fit.params.iloc[1]
        ols_p = fit.pvalues.iloc[1]
        ols_r2 = fit.rsquared
    except Exception:
        ols_beta = np.nan
        ols_p = np.nan
        ols_r2 = np.nan

    try:
        welch_p = ttest_ind(low, delayed, equal_var=False).pvalue
    except Exception:
        welch_p = np.nan

    try:
        mw_p = mannwhitneyu(low, delayed, alternative="two-sided").pvalue
    except Exception:
        mw_p = np.nan

    try:
        rank_p = ranksums(low, delayed).pvalue
    except Exception:
        rank_p = np.nan

    try:
        logit = sm.Logit(df["Binary"], sm.add_constant(df["Score"])).fit(disp=0)
        logit_beta = logit.params.iloc[1]
        logit_p = logit.pvalues.iloc[1]
        logit_or = np.exp(logit_beta)
    except Exception:
        logit_beta = np.nan
        logit_p = np.nan
        logit_or = np.nan

    gdf_used = gdf[gdf["SNP_ID"].isin(variable_snps)].copy()

    n_intergenic = int(gdf_used["Is_intergenic"].sum())
    n_gene_body = int((~gdf_used["Is_intergenic"]).sum())

    intergenic_dist = gdf_used.loc[
        gdf_used["Is_intergenic"],
        "Distance_to_gene_bp"
    ]

    max_dist = intergenic_dist.max() if len(intergenic_dist) > 0 else 0
    mean_dist = intergenic_dist.mean() if len(intergenic_dist) > 0 else 0

    results.append({
        "Gene_ID": gene_id,
        "CHROM": gdf["CHROM"].iloc[0],
        "Gene_mid_POS": gdf["POS"].median(),
        "Min_POS": gdf["POS"].min(),
        "Max_POS": gdf["POS"].max(),
        "N_SNPs_used": len(variable_snps),
        "N_intergenic_SNP_assignments_50kb": n_intergenic,
        "N_gene_body_or_transcript_SNP_assignments": n_gene_body,
        "Max_intergenic_distance_bp": max_dist,
        "Mean_intergenic_distance_bp": mean_dist,
        "SNPs_used": ";".join(variable_snps),
        "Effects_used": ";".join(sorted(gdf_used["Effect"].dropna().unique())),
        "Mean_score_low_PC50_lt_5": low.mean(),
        "Mean_score_delayed_PC50_ge_5": delayed.mean(),
        "Difference_delayed_minus_low": delayed.mean() - low.mean(),
        "OLS_beta_PC50": ols_beta,
        "OLS_p_PC50": ols_p,
        "OLS_R2_PC50": ols_r2,
        "Welch_p": welch_p,
        "MannWhitney_p": mw_p,
        "Wilcoxon_rank_sum_p": rank_p,
        "Logistic_beta": logit_beta,
        "Logistic_p": logit_p,
        "Logistic_OR": logit_or
    })

res = pd.DataFrame(results)

print("\nNumber of tested genes:", res.shape[0])
print("Result columns:", res.columns.tolist())

if res.empty:
    raise ValueError("No genes passed filters.")

# ============================================================
# ADD COMMON NAME
# ============================================================
res = res.merge(gene_names, on="Gene_ID", how="left")

# ============================================================
# FDR
# ============================================================
for pcol in ["OLS_p_PC50", "Welch_p", "MannWhitney_p", "Logistic_p"]:

    res[pcol + "_FDR"] = np.nan

    valid = res[pcol].notna() & (res[pcol] > 0) & (res[pcol] <= 1)

    if valid.sum() > 0:
        res.loc[valid, pcol + "_FDR"] = multipletests(
            res.loc[valid, pcol],
            method="fdr_bh"
        )[1]

res = res.sort_values("OLS_p_PC50", na_position="last").reset_index(drop=True)

# ============================================================
# FINAL TABLE
# ============================================================
final_cols = [
    "Gene_ID",
    "Common_name",
    "CHROM",
    "Gene_mid_POS",
    "Min_POS",
    "Max_POS",
    "N_SNPs_used",
    "N_intergenic_SNP_assignments_50kb",
    "N_gene_body_or_transcript_SNP_assignments",
    "Max_intergenic_distance_bp",
    "Mean_intergenic_distance_bp",
    "Effects_used",
    "Mean_score_low_PC50_lt_5",
    "Mean_score_delayed_PC50_ge_5",
    "Difference_delayed_minus_low",
    "OLS_beta_PC50",
    "OLS_p_PC50",
    "OLS_p_PC50_FDR",
    "OLS_R2_PC50",
    "Welch_p",
    "Welch_p_FDR",
    "MannWhitney_p",
    "MannWhitney_p_FDR",
    "Logistic_p",
    "Logistic_p_FDR",
    "Logistic_OR",
    "SNPs_used"
]

final_table = res[final_cols].copy()

out_csv = "Genomewide_PC50_gene_score_NO_SYNONYMOUS_filtered_FIXED_MANHATTAN.csv"
final_table.to_csv(out_csv, index=False)

print("\nTop 30 genome-wide gene results:")
print(final_table.head(30).to_string(index=False))

print("\nSaved final table:")
print(out_csv)

# ============================================================
# QQ PLOT
# ============================================================
qq = res.dropna(subset=["OLS_p_PC50"]).copy()
qq = qq[(qq["OLS_p_PC50"] > 0) & (qq["OLS_p_PC50"] <= 1)].copy()

pvals = np.sort(qq["OLS_p_PC50"].values)

observed = -np.log10(pvals)
expected = -np.log10(np.arange(1, len(pvals) + 1) / (len(pvals) + 1))

plt.figure(figsize=(6, 6))
plt.scatter(expected, observed, s=25)

max_val = max(expected.max(), observed.max())
plt.plot([0, max_val], [0, max_val], linestyle="--", linewidth=1)

plt.xlabel("Expected -log10(p)")
plt.ylabel("Observed -log10(p)")
plt.title("QQ plot: PC50 gene-score OLS p-values, no synonymous")

plt.tight_layout()
plt.show()

# ============================================================
# MANHATTAN PLOT
# ============================================================
def chrom_number(x):
    x = str(x)
    m = re.search(r"Pf3D7_(\d+)_v3", x)
    return int(m.group(1)) if m else np.nan

man = res.dropna(subset=["OLS_p_PC50", "CHROM", "Gene_mid_POS"]).copy()
man = man[(man["OLS_p_PC50"] > 0) & (man["OLS_p_PC50"] <= 1)].copy()

man["CHR_NUM"] = man["CHROM"].apply(chrom_number)

print("\nRows before Manhattan chromosome filtering:", man.shape[0])
print(man[["Gene_ID", "CHROM", "CHR_NUM", "Gene_mid_POS", "OLS_p_PC50"]].head(20))

man = man.dropna(subset=["CHR_NUM"]).copy()

if man.empty:
    raise ValueError("No valid rows for Manhattan plot after CHROM parsing.")

man["CHR_NUM"] = man["CHR_NUM"].astype(int)
man["Gene_mid_POS"] = pd.to_numeric(man["Gene_mid_POS"], errors="coerce")
man = man.dropna(subset=["Gene_mid_POS"]).copy()

man = man.sort_values(["CHR_NUM", "Gene_mid_POS"]).copy()

chr_centers = {}
offset = 0
cum_positions = []

for chrom in sorted(man["CHR_NUM"].unique()):

    sub = man[man["CHR_NUM"] == chrom].copy()

    x = sub["Gene_mid_POS"] + offset
    cum_positions.extend(x.tolist())

    chr_centers[chrom] = x.mean()

    offset = x.max() + 500000

man["BP_cum"] = cum_positions
man["minus_log10_p"] = -np.log10(man["OLS_p_PC50"])

plt.figure(figsize=(14, 6))

for chrom in sorted(man["CHR_NUM"].unique()):
    sub = man[man["CHR_NUM"] == chrom]

    plt.scatter(
        sub["BP_cum"],
        sub["minus_log10_p"],
        s=30
    )

bonf = 0.05 / man.shape[0]

plt.axhline(-np.log10(bonf), linestyle="--", linewidth=1)
plt.axhline(-np.log10(0.05), linestyle=":", linewidth=1)

plt.xticks(
    [chr_centers[c] for c in sorted(chr_centers)],
    [str(c) for c in sorted(chr_centers)]
)

plt.xlabel("Chromosome")
plt.ylabel("-log10(OLS p-value)")
plt.title("Genome-wide Manhattan plot: PC50 gene score, no synonymous, filtered")

plt.tight_layout()
plt.show()

# ============================================================
# LABELED MANHATTAN
# ============================================================
top_n = 15
top_hits = man.nsmallest(top_n, "OLS_p_PC50").copy()

plt.figure(figsize=(14, 6))

for chrom in sorted(man["CHR_NUM"].unique()):
    sub = man[man["CHR_NUM"] == chrom]

    plt.scatter(
        sub["BP_cum"],
        sub["minus_log10_p"],
        s=30
    )

plt.axhline(-np.log10(bonf), linestyle="--", linewidth=1)
plt.axhline(-np.log10(0.05), linestyle=":", linewidth=1)

for _, row in top_hits.iterrows():

    common_name = str(row["Common_name"]).strip()

    if common_name in ["", "nan", "None"]:
        label = row["Gene_ID"]
    else:
        label = common_name.split(";")[0]

    plt.text(
        row["BP_cum"],
        row["minus_log10_p"] + 0.08,
        label,
        fontsize=8,
        rotation=90,
        ha="left"
    )

plt.xticks(
    [chr_centers[c] for c in sorted(chr_centers)],
    [str(c) for c in sorted(chr_centers)]
)

plt.xlabel("Chromosome")
plt.ylabel("-log10(OLS p-value)")
plt.title("Genome-wide Manhattan plot with top labeled genes: PC50")

plt.tight_layout()
plt.show()

# ============================================================
# SANITY CHECK
# ============================================================
print("\nExamples of closest intergenic SNP assignments within 50 kb:")
if intergenic_df.empty:
    print("No intergenic SNPs were assigned within 50 kb.")
else:
    print(
        intergenic_df[
            ["SNP_ID", "Gene_ID", "Common_name", "Distance_to_gene_bp", "Effect"]
        ]
        .drop_duplicates()
        .head(50)
        .to_string(index=False)
    )

print("\nMaximum intergenic distance included:")
if intergenic_df.empty:
    print("None")
else:
    print(intergenic_df["Distance_to_gene_bp"].max())

print("\nCheck that synonymous variants were removed:")
print(assign["Effect"].astype(str).str.contains("synonymous_variant", na=False).value_counts())

In [ ]:
# ============================================================
# MADSEN-BROWNING 2009 WEIGHTED-SUM TEST
# EXACT PAPER-STYLE FLOW, ADAPTED TO HAPLOID P. falciparum
#
# Paper method:
#   1. Group variants by gene
#   2. Define mutation allele
#   3. Estimate mutation frequency in unaffected/control samples
#   4. Compute variant weights
#   5. Compute per-individual weighted mutation score
#   6. Rank all individuals by score
#   7. Sum ranks in affected/high phenotype group
#   8. Permute affected/unaffected labels
#   9. Get gene-level p-value
#
# For your data:
#   mutation allele = ALT allele
#   haploid genotype = 0/1 ALT absent/present
#   PC50 affected/high = PC50 >= 5
#   RSA affected/high  = RSA >= 1
#
# Outputs:
#   - genome-wide Madsen-Browning results for PC50 and RSA
#   - KIC1 detailed results
#   - Manhattan plots
#   - QQ plots
#   - KIC1 burden/rank/permutation/variant-weight figures
# ============================================================

import os
import re
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import rankdata, norm, mannwhitneyu, ttest_ind, linregress
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

# ============================================================
# 0. Folder setup
# ============================================================

# Use your previous local folder if it exists; otherwise use current folder.
PREFERRED_DIR = Path("/Users/nirjharbhattacharyya/Downloads/Nirjhar_papers_Ferdig_lab/MKN_BD_popgen")

if PREFERRED_DIR.exists():
    BASE_DIR = PREFERRED_DIR
    os.chdir(BASE_DIR)
else:
    BASE_DIR = Path.cwd()

print("Using BASE_DIR:", BASE_DIR)

OUTDIR = BASE_DIR / "KIC1_Madsen_Browning_outputs"
FIGDIR = OUTDIR / "figures"

OUTDIR.mkdir(exist_ok=True)
FIGDIR.mkdir(exist_ok=True)

# ============================================================
# 1. Parameters
# ============================================================

KIC1_GENE = "PF3D7_0606000"

# Exact Madsen-Browning is case/control.
# These thresholds create affected/high vs unaffected/low groups.
PC50_THRESHOLD = 5.0
RSA_THRESHOLD = 1.0

# Paper-style: include observed mutation variants.
# For strict rare-variant analysis, you can later add MAF filters.
MIN_ALT_COUNT = 1
MIN_REF_COUNT = 1
MIN_SNPS_PER_GENE = 1

# For speed:
# genome-wide scan can use fewer permutations;
# KIC1 detail can use more.
GENOME_N_PERM = 2000
KIC1_N_PERM = 10000

RANDOM_SEED = 12345

# Intergenic assignment, same spirit as your earlier code.
MAX_DIST = 50_000

rng = np.random.default_rng(RANDOM_SEED)

# ============================================================
# 2. Find files using your previous file-location logic
# ============================================================

def find_file(patterns):
    for pattern in patterns:
        hits = list(BASE_DIR.glob(pattern))
        if hits:
            print("Found:", hits[0].name)
            return hits[0]

    print("\nFiles in folder:")
    for f in BASE_DIR.iterdir():
        print(" ", f.name)

    raise FileNotFoundError(
        "Could not find file matching any pattern:\n" + "\n".join(patterns)
    )

GENOME_CSV = find_file([
    "ALL_BD_ISOLATE_VCF_TO_CSV*.csv",
    "*BD_ISOLATE*CSV*.csv",
    "*VCF_TO_CSV*.csv",
    "/mnt/data/ALL_BD_ISOLATE_VCF_TO_CSV*.csv",
])

KIC_CSV = find_file([
    "KIC_1_vcf_subset_BD*.csv",
    "*KIC*subset*.csv",
    "*KIC*.csv",
    "/mnt/data/KIC_1_vcf_subset_BD*.csv",
])

print("GENOME_CSV =", GENOME_CSV)
print("KIC_CSV    =", KIC_CSV)

# ============================================================
# 3. Phenotype tables
# ============================================================

pc50 = pd.DataFrame({
    "Sample": [
        "I-001-clone", "I-002", "I-003-clone", "I-004", "I-005",
        "I-008", "I-010", "I-011", "I-013", "I-016",
        "I-017", "I-018", "I-019", "I-020", "I-021",
        "I-025", "I-026", "I-027", "I-029", "I-030",
        "I-031", "I-033", "I-035", "I-036", "I-039",
        "I-040-", "I-041"
    ],
    "PC50": [
        0.015, 3.56, 5.967, 1.6, 3.82,
        2.71, 2.01, 5.35, 7.98, 10.25,
        4.51, 3.42, 2.02, 7.66, 2.83,
        3.39, 3.25, 3.23, 7.68, 8.19,
        3.41, 8.48, 1.25, 9.42, 13.14,
        3.14, 4.82
    ]
})

rsa = pd.DataFrame({
    "Sample": [
        "I-001-clone", "I-003-clone", "I-004", "I-011", "I-016",
        "I-018", "I-020", "I-029", "I-026", "I-033",
        "I-039", "I-041", "I-040", "I-027", "O-024",
        "I-008", "I-002", "O-014", "O-009", "O-032",
        "O-038", "I-019"
    ],
    "RSA": [
        0.6635725467, 2.007394647, 0, 0.6057562733, 4.634666667,
        0.7466666667, 1.76, 6.443333333, 0.6533333333, 1.889666667,
        4.891666667, 1.358666667, 0.7466666667, 0.5593333333, 1.73,
        1.535666667, 2.563333333, 2.126666667, 0.42, 0.783,
        1.371666667, 0.4633333333
    ]
})

def canonical_sample(x):
    x = str(x).strip()
    if x.endswith("-"):
        x = x[:-1]
    return x

pc50["Sample"] = pc50["Sample"].map(canonical_sample)
rsa["Sample"] = rsa["Sample"].map(canonical_sample)

pc50 = pc50.dropna().drop_duplicates("Sample").set_index("Sample")
rsa = rsa.dropna().drop_duplicates("Sample").set_index("Sample")

pc50["Affected"] = (pc50["PC50"] >= PC50_THRESHOLD).astype(int)
rsa["Affected"] = (rsa["RSA"] >= RSA_THRESHOLD).astype(int)

print("\nPC50 phenotype table:")
print(pc50)

print("\nRSA phenotype table:")
print(rsa)

# ============================================================
# 4. Load genome-wide VCF-to-CSV
# ============================================================

print("\nLoading genome CSV...")
geno = pd.read_csv(GENOME_CSV, low_memory=False)
geno.columns = [str(c).strip() for c in geno.columns]

chrom_col = "#CHROM" if "#CHROM" in geno.columns else "CHROM"

required_cols = [chrom_col, "POS", "REF", "ALT", "INFO", "FORMAT"]
missing = [c for c in required_cols if c not in geno.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

geno["POS"] = pd.to_numeric(geno["POS"], errors="coerce")
geno = geno.dropna(subset=["POS"]).copy()
geno["POS"] = geno["POS"].astype(int)

geno["SNP_ID"] = geno.apply(
    lambda r: f"{r[chrom_col]}:{int(r['POS'])}:{r['REF']}>{r['ALT']}",
    axis=1
)

format_idx = list(geno.columns).index("FORMAT")

# Canonicalize sample column names.
rename_map = {
    old: canonical_sample(old)
    for old in geno.columns[(format_idx + 1):]
}
geno = geno.rename(columns=rename_map)

vcf_samples = list(geno.columns[(format_idx + 1):])

print("Genome rows:", geno.shape[0])
print("VCF samples:", len(vcf_samples))

# ============================================================
# 5. Genotype coding: ALT present/absent
# ============================================================

def alt_present(x):
    """
    Haploid parasite genotype coding:
      0 = REF only
      1 = any ALT allele present
      NaN = missing
    """
    if pd.isna(x):
        return np.nan

    gt = str(x).split(":")[0].strip()

    if gt in ["./.", ".|.", ".", "./", "|", "", "nan", "None"]:
        return np.nan

    alleles = re.split(r"[\/|]", gt)

    return 1.0 if any(a not in ["0", "."] for a in alleles) else 0.0

def build_alt_matrix(samples):
    """
    Return matrix:
      rows = samples
      columns = SNP_ID
      values = 0/1 ALT absent/present
    """
    print("Building ALT matrix for", len(samples), "samples...")

    A = pd.DataFrame(
        {
            s: geno[s].map(alt_present).astype(float).values
            for s in samples
        },
        index=geno["SNP_ID"]
    ).T

    return A

# ============================================================
# 6. SNP-to-gene assignment from ANN field
#    Same general assignment logic as your previous pipeline.
# ============================================================

pf_gene_pattern = re.compile(r"PF3D7_\d{7}")

def clean_common_name(x):
    x = str(x).strip()

    if x in ["", ".", "nan", "NaN", "None", "null", "NULL"]:
        return ""

    if pf_gene_pattern.fullmatch(x):
        return ""

    if "_circ" in x:
        return ""

    return x

def is_synonymous_effect(effect):
    return "synonymous_variant" in str(effect)

def build_assignments(geno):
    gene_body_records = []
    intergenic_raw_records = []
    dropped_synonymous = 0

    for _, row in geno.iterrows():
        info = str(row["INFO"])
        snp_id = row["SNP_ID"]
        chrom = row[chrom_col]
        pos = int(row["POS"])

        ann_field = None
        for field in info.split(";"):
            if field.startswith("ANN="):
                ann_field = field[4:]
                break

        if ann_field is None:
            continue

        for ann_record in ann_field.split(","):
            parts = ann_record.split("|")

            if len(parts) < 7:
                continue

            effect = parts[1]
            impact = parts[2]
            gene_name_field = parts[3]
            gene_id_field = parts[4]
            feature_type = parts[5]
            feature_id = parts[6]

            if is_synonymous_effect(effect):
                dropped_synonymous += 1
                continue

            text = "|".join([gene_id_field, feature_id, gene_name_field, ann_record])
            gene_ids = sorted(set(pf_gene_pattern.findall(text)))

            if not gene_ids:
                continue

            if "intergenic_region" in effect:
                raw_common_names = str(gene_name_field).split("-")
                clean_names = [clean_common_name(x) for x in raw_common_names]
                clean_names = [x for x in clean_names if x != ""]

                intergenic_raw_records.append({
                    "SNP_ID": snp_id,
                    "CHROM": chrom,
                    "POS": pos,
                    "REF": row["REF"],
                    "ALT": row["ALT"],
                    "Candidate_Gene_IDs": gene_ids,
                    "Candidate_common_names": clean_names,
                    "Effect": effect,
                    "Impact": impact,
                    "Feature_type": feature_type,
                    "Feature_ID": feature_id,
                })

            else:
                common_name = clean_common_name(gene_name_field)

                for gene_id in gene_ids:
                    gene_body_records.append({
                        "SNP_ID": snp_id,
                        "Gene_ID": gene_id,
                        "Common_name": common_name,
                        "CHROM": chrom,
                        "POS": pos,
                        "REF": row["REF"],
                        "ALT": row["ALT"],
                        "Effect": effect,
                        "Impact": impact,
                        "Feature_type": feature_type,
                        "Feature_ID": feature_id,
                        "Is_intergenic": False,
                        "Distance_to_gene_bp": 0.0,
                        "Assignment_rule": "gene_body_or_transcript_non_synonymous",
                    })

    gene_body_df = pd.DataFrame(gene_body_records)
    intergenic_raw_df = pd.DataFrame(intergenic_raw_records)

    if gene_body_df.empty:
        raise ValueError("No non-synonymous gene-body/transcript annotations found.")

    gene_intervals = (
        gene_body_df
        .groupby("Gene_ID")
        .agg(
            CHROM=("CHROM", "first"),
            Gene_start_observed=("POS", "min"),
            Gene_end_observed=("POS", "max"),
            Gene_mid_observed=("POS", "median")
        )
        .reset_index()
    )

    gene_interval_dict = gene_intervals.set_index("Gene_ID").to_dict("index")

    intergenic_assigned_records = []

    for _, row in intergenic_raw_df.iterrows():
        snp_pos = int(row["POS"])
        snp_chrom = row["CHROM"]

        candidate_rows = []

        for gene_id in row["Candidate_Gene_IDs"]:
            if gene_id not in gene_interval_dict:
                continue

            g = gene_interval_dict[gene_id]

            if g["CHROM"] != snp_chrom:
                continue

            gene_start = int(g["Gene_start_observed"])
            gene_end = int(g["Gene_end_observed"])

            if snp_pos < gene_start:
                dist = gene_start - snp_pos
            elif snp_pos > gene_end:
                dist = snp_pos - gene_end
            else:
                dist = 0

            candidate_rows.append({
                "Gene_ID": gene_id,
                "Distance_to_gene_bp": dist,
                "Gene_start_observed": gene_start,
                "Gene_end_observed": gene_end
            })

        if not candidate_rows:
            continue

        candidate_df = pd.DataFrame(candidate_rows)
        min_dist = candidate_df["Distance_to_gene_bp"].min()

        if min_dist > MAX_DIST:
            continue

        closest_genes = candidate_df[
            candidate_df["Distance_to_gene_bp"] == min_dist
        ].copy()

        for _, closest in closest_genes.iterrows():
            gene_id = closest["Gene_ID"]

            common_name = ""

            if len(row["Candidate_common_names"]) == len(row["Candidate_Gene_IDs"]):
                gene_to_name = dict(zip(row["Candidate_Gene_IDs"], row["Candidate_common_names"]))
                common_name = gene_to_name.get(gene_id, "")

            if common_name == "":
                common_name = ";".join(row["Candidate_common_names"])

            intergenic_assigned_records.append({
                "SNP_ID": row["SNP_ID"],
                "Gene_ID": gene_id,
                "Common_name": common_name,
                "CHROM": row["CHROM"],
                "POS": row["POS"],
                "REF": row["REF"],
                "ALT": row["ALT"],
                "Effect": row["Effect"],
                "Impact": row["Impact"],
                "Feature_type": row["Feature_type"],
                "Feature_ID": row["Feature_ID"],
                "Is_intergenic": True,
                "Distance_to_gene_bp": float(min_dist),
                "Assignment_rule": "closest_intergenic_candidate_within_50kb",
            })

    intergenic_df = pd.DataFrame(intergenic_assigned_records)

    assign = pd.concat([gene_body_df, intergenic_df], ignore_index=True)
    assign = assign.drop_duplicates(
        subset=["SNP_ID", "Gene_ID", "Effect", "Is_intergenic"]
    ).copy()

    assign = assign[
        ~assign["Effect"].astype(str).str.contains("synonymous_variant", na=False)
    ].copy()

    return assign, gene_body_df, intergenic_raw_df, intergenic_df, dropped_synonymous

assign_path = OUTDIR / "SNP_to_gene_assignments_no_synonymous_closest_intergenic_50kb.csv"

if assign_path.exists():
    print("Loading existing assignment file...")
    assign = pd.read_csv(assign_path)
    dropped_synonymous = np.nan
else:
    print("Building SNP-to-gene assignments...")
    assign, gene_body_df, intergenic_raw_df, intergenic_df, dropped_synonymous = build_assignments(geno)
    assign.to_csv(assign_path, index=False)

print("Assignments:", assign.shape)
print("Unique genes:", assign["Gene_ID"].nunique())

def collapse_names(x):
    vals = []

    for item in x.dropna().astype(str):
        for z in item.split(";"):
            z = clean_common_name(z)
            if z:
                vals.append(z)

    return ";".join(sorted(set(vals)))

gene_names = (
    assign
    .groupby("Gene_ID")["Common_name"]
    .apply(collapse_names)
    .reset_index()
)

# ============================================================
# 7. Madsen-Browning exact weighted-sum functions
# ============================================================

def mb_compute_weights_haploid(G, affected):
    """
    Madsen-Browning weighting adapted to haploid ALT presence.

    Original paper estimates mutation frequency from unaffected individuals.
    For haploid parasite data:
        q_j = (m_U + 1) / (n_U + 2)

    Then the scaling term is:
        sqrt(n_total * q_j * (1 - q_j))

    The weighted contribution is:
        ALT_ij / sqrt(n_total * q_j * (1 - q_j))

    Parameters
    ----------
    G : n_samples x n_variants matrix, values 0/1/NaN
    affected : binary vector, 1 = affected/high phenotype, 0 = unaffected/low phenotype

    Returns
    -------
    weights : variant scaling weights used in weighted sum
    q : estimated mutation frequency in unaffected group
    scale : denominator sqrt(n_total*q*(1-q))
    """
    G = np.asarray(G, dtype=float)
    affected = np.asarray(affected, dtype=int)

    controls = affected == 0

    n_variants = G.shape[1]

    q = np.zeros(n_variants)
    scale = np.zeros(n_variants)
    weights = np.zeros(n_variants)

    for j in range(n_variants):
        gj = G[:, j]

        ctrl_nonmissing = controls & ~np.isnan(gj)
        all_nonmissing = ~np.isnan(gj)

        m_u = np.nansum(gj[ctrl_nonmissing])
        n_u = np.sum(ctrl_nonmissing)
        n_total = np.sum(all_nonmissing)

        if n_u == 0 or n_total == 0:
            q[j] = np.nan
            scale[j] = np.nan
            weights[j] = np.nan
            continue

        # Haploid version of the paper's +1/+2 smoothing.
        q_j = (m_u + 1.0) / (n_u + 2.0)

        # Avoid numerical extremes.
        q_j = min(max(q_j, 1e-8), 1 - 1e-8)

        scale_j = math.sqrt(n_total * q_j * (1.0 - q_j))

        q[j] = q_j
        scale[j] = scale_j
        weights[j] = 1.0 / scale_j

    return weights, q, scale

def mb_genetic_score(G, weights):
    """
    Per-sample Madsen-Browning genetic score:
        gamma_i = sum_j ALT_ij * weight_j

    Missing genotypes are treated as zero contribution.
    """
    G = np.asarray(G, dtype=float)
    W = np.asarray(weights, dtype=float)

    G2 = np.where(np.isnan(G), 0.0, G)
    W2 = np.where(np.isnan(W), 0.0, W)

    return G2 @ W2

def mb_rank_sum_stat(scores, affected):
    """
    Rank all samples by genetic score and sum ranks among affected samples.
    """
    affected = np.asarray(affected, dtype=int)

    ranks = rankdata(scores, method="average")
    x = np.sum(ranks[affected == 1])

    return x, ranks

def mb_weighted_sum_test(
    G,
    affected,
    n_perm=2000,
    seed=12345,
    recompute_weights_each_permutation=True,
):
    """
    Exact Madsen-Browning-style test:
      - compute weights from unaffected controls
      - compute genetic scores
      - rank samples
      - sum ranks among affected
      - permute affected labels
      - repeat weighting/scoring/ranking
      - compute standardized z and direct permutation p-value

    One-sided test:
      affected/high phenotype group has higher mutation burden.
    """
    rng_local = np.random.default_rng(seed)

    G = np.asarray(G, dtype=float)
    affected = np.asarray(affected, dtype=int)

    n = len(affected)

    if np.sum(affected == 1) == 0 or np.sum(affected == 0) == 0:
        raise ValueError("Need both affected and unaffected samples.")

    weights_obs, q_obs, scale_obs = mb_compute_weights_haploid(G, affected)
    scores_obs = mb_genetic_score(G, weights_obs)
    x_obs, ranks_obs = mb_rank_sum_stat(scores_obs, affected)

    x_perm = np.zeros(n_perm)

    for b in range(n_perm):
        affected_perm = rng_local.permutation(affected)

        if recompute_weights_each_permutation:
            weights_perm, _, _ = mb_compute_weights_haploid(G, affected_perm)
            scores_perm = mb_genetic_score(G, weights_perm)
        else:
            scores_perm = scores_obs.copy()

        x_b, _ = mb_rank_sum_stat(scores_perm, affected_perm)
        x_perm[b] = x_b

    perm_mean = float(np.mean(x_perm))
    perm_sd = float(np.std(x_perm, ddof=1))

    if perm_sd > 0:
        z = (x_obs - perm_mean) / perm_sd
        p_norm = float(1.0 - norm.cdf(z))
    else:
        z = np.nan
        p_norm = np.nan

    # Direct one-sided permutation p-value:
    # affected group has higher scores/ranks.
    p_perm = float((1.0 + np.sum(x_perm >= x_obs)) / (n_perm + 1.0))

    out = {
        "x_obs": float(x_obs),
        "x_perm": x_perm,
        "perm_mean": perm_mean,
        "perm_sd": perm_sd,
        "z": z,
        "p_norm_one_sided": p_norm,
        "p_perm_one_sided": p_perm,
        "scores": scores_obs,
        "ranks": ranks_obs,
        "weights": weights_obs,
        "q_unaffected": q_obs,
        "scale": scale_obs,
        "n_perm": n_perm,
    }

    return out

# ============================================================
# 8. Gene-level scan
# ============================================================

def scan_phenotype_mb(pheno_df, phenotype_col, n_perm=GENOME_N_PERM):
    """
    Genome-wide Madsen-Browning scan for one binary affected phenotype.
    """
    common = sorted(set(vcf_samples).intersection(set(pheno_df.index)))

    if len(common) < 5:
        raise ValueError(f"Too few matched samples for {phenotype_col}: {len(common)}")

    print(f"\nScanning {phenotype_col}")
    print("Matched samples:", len(common))
    print(common)

    A = build_alt_matrix(common)

    y_cont = pheno_df.loc[common, phenotype_col].astype(float).values
    affected = pheno_df.loc[common, "Affected"].astype(int).values

    results = []

    seed_base = 7000 if phenotype_col == "PC50" else 8000

    for gi, (gene_id, gdf) in enumerate(assign.groupby("Gene_ID")):
        snps_all = sorted(set(gdf["SNP_ID"]).intersection(A.columns))

        if len(snps_all) < MIN_SNPS_PER_GENE:
            continue

        A_gene = A[snps_all].copy()

        alt_counts = A_gene.sum(axis=0, skipna=True)
        nonmissing_counts = A_gene.notna().sum(axis=0)
        ref_counts = nonmissing_counts - alt_counts

        keep_snps = alt_counts[
            (alt_counts >= MIN_ALT_COUNT) &
            (ref_counts >= MIN_REF_COUNT)
        ].index.tolist()

        if len(keep_snps) < MIN_SNPS_PER_GENE:
            continue

        A_gene = A_gene[keep_snps].copy()

        # Remove invariant after filtering.
        freq = A_gene.mean(axis=0, skipna=True)
        variable_snps = freq[(freq > 0) & (freq < 1)].index.tolist()

        if len(variable_snps) < MIN_SNPS_PER_GENE:
            continue

        A_gene = A_gene[variable_snps].copy()

        try:
            stat = mb_weighted_sum_test(
                A_gene.values,
                affected,
                n_perm=n_perm,
                seed=seed_base + gi,
                recompute_weights_each_permutation=True
            )
        except Exception:
            continue

        gdf_used = gdf[gdf["SNP_ID"].isin(variable_snps)].copy()

        n_intergenic = int(gdf_used["Is_intergenic"].sum()) if "Is_intergenic" in gdf_used else 0
        n_gene_body = int((~gdf_used["Is_intergenic"]).sum()) if "Is_intergenic" in gdf_used else len(gdf_used)

        intergenic_dist = (
            gdf_used.loc[gdf_used["Is_intergenic"], "Distance_to_gene_bp"]
            if "Is_intergenic" in gdf_used else pd.Series(dtype=float)
        )

        # Simple continuous descriptive association with MB score.
        if len(np.unique(stat["scores"])) > 1:
            lr = linregress(stat["scores"], y_cont)
            score_cont_slope = float(lr.slope)
            score_cont_p = float(lr.pvalue)
            score_cont_r2 = float(lr.rvalue ** 2)
        else:
            score_cont_slope = np.nan
            score_cont_p = np.nan
            score_cont_r2 = np.nan

        results.append({
            "Gene_ID": gene_id,
            "CHROM": gdf["CHROM"].iloc[0],
            "Gene_mid_POS": float(gdf["POS"].median()),
            "Min_POS": int(gdf["POS"].min()),
            "Max_POS": int(gdf["POS"].max()),
            "Phenotype": phenotype_col,
            "Threshold_for_affected": PC50_THRESHOLD if phenotype_col == "PC50" else RSA_THRESHOLD,
            "N_samples": len(common),
            "N_affected": int(np.sum(affected == 1)),
            "N_unaffected": int(np.sum(affected == 0)),
            "N_SNPs_total_assigned": len(snps_all),
            "N_SNPs_used": len(variable_snps),
            "N_intergenic_SNP_assignments_50kb": n_intergenic,
            "N_gene_body_or_transcript_SNP_assignments": n_gene_body,
            "Max_intergenic_distance_bp": float(intergenic_dist.max()) if len(intergenic_dist) else 0.0,
            "Mean_intergenic_distance_bp": float(intergenic_dist.mean()) if len(intergenic_dist) else 0.0,
            "Effects_used": ";".join(sorted(gdf_used["Effect"].dropna().astype(str).unique())),
            "SNPs_used": ";".join(variable_snps),
            "MB_rank_sum_observed": stat["x_obs"],
            "MB_perm_mean": stat["perm_mean"],
            "MB_perm_sd": stat["perm_sd"],
            "MB_z": stat["z"],
            "MB_p_norm_one_sided": stat["p_norm_one_sided"],
            "MB_p_perm_one_sided": stat["p_perm_one_sided"],
            "MB_n_perm": n_perm,
            "Mean_score_affected": float(np.mean(stat["scores"][affected == 1])),
            "Mean_score_unaffected": float(np.mean(stat["scores"][affected == 0])),
            "Median_score_affected": float(np.median(stat["scores"][affected == 1])),
            "Median_score_unaffected": float(np.median(stat["scores"][affected == 0])),
            "Continuous_score_slope": score_cont_slope,
            "Continuous_score_p": score_cont_p,
            "Continuous_score_R2": score_cont_r2,
        })

        if (gi + 1) % 500 == 0:
            print(phenotype_col, "gene groups processed:", gi + 1, "results:", len(results))

    res = pd.DataFrame(results)

    if res.empty:
        raise ValueError(f"No genes passed filters for {phenotype_col}")

    res = res.merge(gene_names, on="Gene_ID", how="left")

    valid = (
        res["MB_p_perm_one_sided"].notna() &
        (res["MB_p_perm_one_sided"] > 0) &
        (res["MB_p_perm_one_sided"] <= 1)
    )

    res["MB_FDR_BH"] = np.nan
    res.loc[valid, "MB_FDR_BH"] = multipletests(
        res.loc[valid, "MB_p_perm_one_sided"],
        method="fdr_bh"
    )[1]

    res["MB_Bonferroni"] = np.nan
    res.loc[valid, "MB_Bonferroni"] = np.minimum(
        res.loc[valid, "MB_p_perm_one_sided"] * valid.sum(),
        1.0
    )

    res = res.sort_values("MB_p_perm_one_sided").reset_index(drop=True)

    return res, A, common, y_cont, affected

# ============================================================
# 9. Run genome-wide scans
# ============================================================

pc50_res, A_pc50, common_pc50, y_pc50, aff_pc50 = scan_phenotype_mb(
    pc50,
    "PC50",
    n_perm=GENOME_N_PERM
)

pc50_res.to_csv(
    OUTDIR / "PC50_genomewide_Madsen_Browning_results.csv",
    index=False
)

print("\nTop PC50 Madsen-Browning hits:")
print(
    pc50_res.head(10)[[
        "Gene_ID", "Common_name", "CHROM", "N_SNPs_used",
        "MB_z", "MB_p_perm_one_sided", "MB_FDR_BH"
    ]].to_string(index=False)
)

rsa_res, A_rsa, common_rsa, y_rsa, aff_rsa = scan_phenotype_mb(
    rsa,
    "RSA",
    n_perm=GENOME_N_PERM
)

rsa_res.to_csv(
    OUTDIR / "RSA_genomewide_Madsen_Browning_results.csv",
    index=False
)

print("\nTop RSA Madsen-Browning hits:")
print(
    rsa_res.head(10)[[
        "Gene_ID", "Common_name", "CHROM", "N_SNPs_used",
        "MB_z", "MB_p_perm_one_sided", "MB_FDR_BH"
    ]].to_string(index=False)
)

# ============================================================
# 10. KIC1 detailed analysis
# ============================================================

def detail_gene_mb(gene_id, A, common, y_cont, affected, phenotype_col, n_perm=KIC1_N_PERM):
    gdf = assign[assign["Gene_ID"] == gene_id].copy()

    snps_all = sorted(set(gdf["SNP_ID"]).intersection(A.columns))

    if len(snps_all) < MIN_SNPS_PER_GENE:
        raise ValueError(f"{gene_id} has too few assigned SNPs.")

    A_gene = A[snps_all].copy()

    alt_counts = A_gene.sum(axis=0, skipna=True)
    nonmissing_counts = A_gene.notna().sum(axis=0)
    ref_counts = nonmissing_counts - alt_counts

    keep_snps = alt_counts[
        (alt_counts >= MIN_ALT_COUNT) &
        (ref_counts >= MIN_REF_COUNT)
    ].index.tolist()

    A_gene = A_gene[keep_snps].copy()

    freq = A_gene.mean(axis=0, skipna=True)
    variable_snps = freq[(freq > 0) & (freq < 1)].index.tolist()

    A_gene = A_gene[variable_snps].copy()

    if A_gene.shape[1] < MIN_SNPS_PER_GENE:
        raise ValueError(f"{gene_id} failed SNP filters for {phenotype_col}.")

    stat = mb_weighted_sum_test(
        A_gene.values,
        affected,
        n_perm=n_perm,
        seed=777 if phenotype_col == "PC50" else 778,
        recompute_weights_each_permutation=True
    )

    scores = stat["scores"]
    ranks = stat["ranks"]

    if len(np.unique(scores)) > 1:
        lr = linregress(scores, y_cont)
        cont_slope = float(lr.slope)
        cont_p = float(lr.pvalue)
        cont_r2 = float(lr.rvalue ** 2)
    else:
        cont_slope = np.nan
        cont_p = np.nan
        cont_r2 = np.nan

    try:
        mw = mannwhitneyu(
            scores[affected == 1],
            scores[affected == 0],
            alternative="greater"
        )
        mw_p = float(mw.pvalue)
    except Exception:
        mw_p = np.nan

    score_table = pd.DataFrame({
        "Sample": common,
        "Phenotype": y_cont,
        "Affected": affected,
        "Affected_label": np.where(affected == 1, "Affected/high", "Unaffected/low"),
        "MB_weighted_score": scores,
        "MB_rank": ranks,
    })

    per_variant = pd.DataFrame({
        "SNP_ID": variable_snps,
        "ALT_count": A_gene.sum(axis=0, skipna=True).values,
        "REF_count": (A_gene.notna().sum(axis=0) - A_gene.sum(axis=0, skipna=True)).values,
        "ALT_frequency_all": A_gene.mean(axis=0, skipna=True).values,
        "q_unaffected_smoothed": stat["q_unaffected"],
        "MB_scale_sqrt_nq1q": stat["scale"],
        "MB_weight_1_over_scale": stat["weights"],
    })

    per_variant = per_variant.merge(
        assign[[
            "SNP_ID", "Gene_ID", "CHROM", "POS", "REF", "ALT",
            "Effect", "Impact", "Is_intergenic", "Distance_to_gene_bp"
        ]].drop_duplicates("SNP_ID"),
        on="SNP_ID",
        how="left"
    )

    summary = {
        "Gene_ID": gene_id,
        "Phenotype": phenotype_col,
        "Threshold_for_affected": PC50_THRESHOLD if phenotype_col == "PC50" else RSA_THRESHOLD,
        "N_samples": len(common),
        "N_affected": int(np.sum(affected == 1)),
        "N_unaffected": int(np.sum(affected == 0)),
        "Samples": ";".join(common),
        "N_SNPs_used": A_gene.shape[1],
        "MB_rank_sum_observed": stat["x_obs"],
        "MB_perm_mean": stat["perm_mean"],
        "MB_perm_sd": stat["perm_sd"],
        "MB_z": stat["z"],
        "MB_p_norm_one_sided": stat["p_norm_one_sided"],
        "MB_p_perm_one_sided": stat["p_perm_one_sided"],
        "MB_n_perm": n_perm,
        "Mean_score_affected": float(np.mean(scores[affected == 1])),
        "Mean_score_unaffected": float(np.mean(scores[affected == 0])),
        "Median_score_affected": float(np.median(scores[affected == 1])),
        "Median_score_unaffected": float(np.median(scores[affected == 0])),
        "Mann_Whitney_greater_p": mw_p,
        "Continuous_score_slope": cont_slope,
        "Continuous_score_p": cont_p,
        "Continuous_score_R2": cont_r2,
    }

    return summary, score_table, per_variant, stat, A_gene

kic_pc50_summary, kic_pc50_scores, kic_pc50_var, kic_pc50_stat, kic_pc50_G = detail_gene_mb(
    KIC1_GENE,
    A_pc50,
    common_pc50,
    y_pc50,
    aff_pc50,
    "PC50",
    n_perm=KIC1_N_PERM
)

kic_rsa_summary, kic_rsa_scores, kic_rsa_var, kic_rsa_stat, kic_rsa_G = detail_gene_mb(
    KIC1_GENE,
    A_rsa,
    common_rsa,
    y_rsa,
    aff_rsa,
    "RSA",
    n_perm=KIC1_N_PERM
)

kic_summary_df = pd.DataFrame([kic_pc50_summary, kic_rsa_summary])

kic_summary_df.to_csv(
    OUTDIR / "KIC1_Madsen_Browning_detail_summary.csv",
    index=False
)

kic_pc50_scores.to_csv(
    OUTDIR / "KIC1_PC50_Madsen_Browning_sample_scores.csv",
    index=False
)

kic_rsa_scores.to_csv(
    OUTDIR / "KIC1_RSA_Madsen_Browning_sample_scores.csv",
    index=False
)

kic_pc50_var.to_csv(
    OUTDIR / "KIC1_PC50_Madsen_Browning_variant_weights.csv",
    index=False
)

kic_rsa_var.to_csv(
    OUTDIR / "KIC1_RSA_Madsen_Browning_variant_weights.csv",
    index=False
)

print("\nKIC1 Madsen-Browning summary:")
print(kic_summary_df.to_string(index=False))

# ============================================================
# 11. Plotting functions
# ============================================================

def chrom_number(x):
    m = re.search(r"Pf3D7_(\d+)_v3", str(x))
    return int(m.group(1)) if m else np.nan

def savefig(path_base):
    plt.savefig(str(path_base) + ".png", dpi=300, bbox_inches="tight")
    plt.savefig(str(path_base) + ".pdf", bbox_inches="tight")
    plt.close()

def qq_plot_mb(res, phenotype_col):
    p = res["MB_p_perm_one_sided"].dropna()
    p = p[(p > 0) & (p <= 1)].sort_values().values

    obs = -np.log10(p)
    exp = -np.log10(np.arange(1, len(p) + 1) / (len(p) + 1))

    plt.figure(figsize=(6, 6))
    plt.scatter(exp, obs, s=18)

    m = max(float(exp.max()), float(obs.max())) if len(p) else 1
    plt.plot([0, m], [0, m], linestyle="--", linewidth=1)

    plt.xlabel("Expected -log10(p)")
    plt.ylabel("Observed -log10(p)")
    plt.title(f"{phenotype_col} Madsen-Browning QQ plot")
    plt.tight_layout()

    savefig(FIGDIR / f"{phenotype_col}_Madsen_Browning_QQ")

def manhattan_plot_mb(res, phenotype_col, label_top=12):
    man = res.dropna(subset=["MB_p_perm_one_sided", "CHROM", "Gene_mid_POS"]).copy()
    man = man[
        (man["MB_p_perm_one_sided"] > 0) &
        (man["MB_p_perm_one_sided"] <= 1)
    ].copy()

    man["CHR_NUM"] = man["CHROM"].map(chrom_number)
    man = man.dropna(subset=["CHR_NUM"]).copy()
    man["CHR_NUM"] = man["CHR_NUM"].astype(int)

    man = man.sort_values(["CHR_NUM", "Gene_mid_POS"]).copy()

    offset = 0
    centers = {}
    chunks = []

    for chrom in sorted(man["CHR_NUM"].unique()):
        sub = man[man["CHR_NUM"] == chrom].copy()
        sub["BP_cum"] = sub["Gene_mid_POS"] + offset
        centers[chrom] = sub["BP_cum"].mean()
        offset = sub["BP_cum"].max() + 500000
        chunks.append(sub)

    man = pd.concat(chunks, ignore_index=True)
    man["minus_log10_p"] = -np.log10(man["MB_p_perm_one_sided"])

    plt.figure(figsize=(14, 6))

    for chrom in sorted(man["CHR_NUM"].unique()):
        sub = man[man["CHR_NUM"] == chrom]
        plt.scatter(sub["BP_cum"], sub["minus_log10_p"], s=20)

    bonf = 0.05 / len(man)
    plt.axhline(-np.log10(bonf), linestyle="--", linewidth=1, label="Bonferroni 0.05")

    plt.xticks(
        [centers[c] for c in sorted(centers)],
        [str(c) for c in sorted(centers)]
    )

    plt.xlabel("Chromosome")
    plt.ylabel("-log10(Madsen-Browning permutation p)")
    plt.title(f"{phenotype_col} genome-wide Madsen-Browning weighted-sum test")

    top = man.nsmallest(label_top, "MB_p_perm_one_sided")

    for _, row in top.iterrows():
        label = str(row.get("Common_name", "")).split(";")[0]
        if label in ["", "nan", "None"]:
            label = row["Gene_ID"]

        plt.text(
            row["BP_cum"],
            row["minus_log10_p"] + 0.05,
            label,
            fontsize=7,
            rotation=75,
            ha="left"
        )

    plt.tight_layout()
    savefig(FIGDIR / f"{phenotype_col}_Madsen_Browning_Manhattan")

def top_hits_barplot(res, phenotype_col, n=20):
    top = res.nsmallest(n, "MB_p_perm_one_sided").copy()
    top["label"] = top["Common_name"].fillna("").astype(str)

    top.loc[top["label"].isin(["", "nan", "None"]), "label"] = top.loc[
        top["label"].isin(["", "nan", "None"]),
        "Gene_ID"
    ]

    top["minus_log10_p"] = -np.log10(top["MB_p_perm_one_sided"])

    plt.figure(figsize=(10, max(5, n * 0.3)))
    plt.barh(top["label"][::-1], top["minus_log10_p"][::-1])
    plt.xlabel("-log10(Madsen-Browning permutation p)")
    plt.ylabel("Gene")
    plt.title(f"Top {n} {phenotype_col} Madsen-Browning genes")
    plt.tight_layout()

    savefig(FIGDIR / f"{phenotype_col}_Madsen_Browning_top{n}_barplot")

def kic_detail_plots(phenotype_col, score_df, per_variant, stat, G, threshold):
    # --------------------------------------------------------
    # 1. KIC1 score by affected/unaffected group
    # --------------------------------------------------------
    groups = ["Unaffected/low", "Affected/high"]
    data = [
        score_df.loc[score_df["Affected_label"] == g, "MB_weighted_score"].values
        for g in groups
    ]

    plt.figure(figsize=(7, 5))
    plt.violinplot(data, showmeans=True, showmedians=True)
    plt.boxplot(data, widths=0.15)

    for i, g in enumerate(groups, start=1):
        vals = score_df.loc[score_df["Affected_label"] == g, "MB_weighted_score"].values
        jitter = np.random.default_rng(1).normal(i, 0.035, size=len(vals))
        plt.scatter(jitter, vals, s=45, alpha=0.85)

    plt.xticks([1, 2], groups)
    plt.ylabel("KIC1 Madsen-Browning weighted ALT score")
    plt.title(f"KIC1 {phenotype_col}: weighted burden score by group")
    plt.tight_layout()

    savefig(FIGDIR / f"KIC1_{phenotype_col}_MB_score_violin")

    # --------------------------------------------------------
    # 2. Ranked scores, paper-style visualization
    # --------------------------------------------------------
    ranked = score_df.sort_values("MB_rank").copy()

    plt.figure(figsize=(10, 5))
    x = np.arange(len(ranked))

    plt.scatter(
        x,
        ranked["MB_weighted_score"],
        s=70,
        c=ranked["Affected"],
        alpha=0.9
    )

    for xi, (_, row) in zip(x, ranked.iterrows()):
        plt.text(
            xi,
            row["MB_weighted_score"],
            row["Sample"],
            fontsize=8,
            rotation=45,
            ha="left",
            va="bottom"
        )

    plt.xlabel("Samples ranked by Madsen-Browning genetic score")
    plt.ylabel("KIC1 weighted ALT score")
    plt.title(f"KIC1 {phenotype_col}: ranked genetic scores\nAffected/high colored as 1")
    plt.tight_layout()

    savefig(FIGDIR / f"KIC1_{phenotype_col}_MB_ranked_scores")

    # --------------------------------------------------------
    # 3. Permutation null distribution
    # --------------------------------------------------------
    plt.figure(figsize=(7, 5))
    plt.hist(stat["x_perm"], bins=35, alpha=0.85)
    plt.axvline(stat["x_obs"], linestyle="--", linewidth=2)

    plt.xlabel("Affected-rank-sum statistic under permutation")
    plt.ylabel("Count")
    plt.title(
        f"KIC1 {phenotype_col}: Madsen-Browning permutation null\n"
        f"p={stat['p_perm_one_sided']:.4g}, z={stat['z']:.2f}"
    )
    plt.tight_layout()

    savefig(FIGDIR / f"KIC1_{phenotype_col}_MB_permutation_null")

    # --------------------------------------------------------
    # 4. Variant weights
    # --------------------------------------------------------
    plot_var = per_variant.copy()
    plot_var["POS_label"] = plot_var["POS"].astype(str)

    plt.figure(figsize=(10, 4))
    plt.bar(plot_var["POS_label"], plot_var["MB_weight_1_over_scale"])
    plt.xticks(rotation=75)
    plt.xlabel("KIC1 variant position")
    plt.ylabel("Madsen-Browning weight")
    plt.title(f"KIC1 {phenotype_col}: variant weights")
    plt.tight_layout()

    savefig(FIGDIR / f"KIC1_{phenotype_col}_MB_variant_weights")

    # --------------------------------------------------------
    # 5. Genotype heatmap
    # --------------------------------------------------------
    plt.figure(figsize=(10, max(3.5, G.shape[0] * 0.3)))
    im = plt.imshow(G.values, aspect="auto", interpolation="nearest")

    plt.yticks(np.arange(G.shape[0]), G.index, fontsize=8)

    xticklabels = []
    for snp in G.columns:
        match_pos = per_variant.loc[per_variant["SNP_ID"] == snp, "POS"]
        if len(match_pos):
            xticklabels.append(str(int(match_pos.iloc[0])))
        else:
            xticklabels.append(snp)

    plt.xticks(np.arange(G.shape[1]), xticklabels, rotation=75, fontsize=8)
    plt.xlabel("KIC1 variant position")
    plt.ylabel("Sample")
    plt.title(f"KIC1 {phenotype_col}: ALT-present genotype matrix")
    plt.colorbar(im, label="ALT present")
    plt.tight_layout()

    savefig(FIGDIR / f"KIC1_{phenotype_col}_MB_genotype_heatmap")

    # --------------------------------------------------------
    # 6. Continuous phenotype scatter, descriptive only
    # --------------------------------------------------------
    x = score_df["MB_weighted_score"].astype(float).values
    y = score_df["Phenotype"].astype(float).values

    plt.figure(figsize=(8, 6))
    plt.scatter(x, y, s=80, alpha=0.9)

    for _, row in score_df.iterrows():
        plt.text(
            row["MB_weighted_score"],
            row["Phenotype"],
            row["Sample"],
            fontsize=8,
            ha="left",
            va="bottom"
        )

    if len(np.unique(x)) > 1:
        lr = linregress(x, y)
        xline = np.linspace(np.min(x), np.max(x), 200)
        yline = lr.intercept + lr.slope * xline
        plt.plot(xline, yline, linewidth=2)
        title_extra = f"OLS descriptive p={lr.pvalue:.3g}, R²={lr.rvalue**2:.2f}"
    else:
        title_extra = "OLS not available: constant score"

    plt.axhline(threshold, linestyle="--", linewidth=1.5)

    plt.xlabel("KIC1 Madsen-Browning weighted ALT score")
    plt.ylabel(phenotype_col)
    plt.title(f"KIC1 {phenotype_col}: score vs continuous phenotype\n{title_extra}")
    plt.tight_layout()

    savefig(FIGDIR / f"KIC1_{phenotype_col}_MB_score_vs_continuous_phenotype")

# ============================================================
# 12. Generate all figures
# ============================================================

qq_plot_mb(pc50_res, "PC50")
qq_plot_mb(rsa_res, "RSA")

manhattan_plot_mb(pc50_res, "PC50")
manhattan_plot_mb(rsa_res, "RSA")

top_hits_barplot(pc50_res, "PC50", n=20)
top_hits_barplot(rsa_res, "RSA", n=20)

kic_detail_plots(
    "PC50",
    kic_pc50_scores,
    kic_pc50_var,
    kic_pc50_stat,
    kic_pc50_G,
    threshold=PC50_THRESHOLD
)

kic_detail_plots(
    "RSA",
    kic_rsa_scores,
    kic_rsa_var,
    kic_rsa_stat,
    kic_rsa_G,
    threshold=RSA_THRESHOLD
)

# ============================================================
# 13. Save summary files
# ============================================================

summary_lines = []

summary_lines.append("Madsen-Browning weighted-sum analysis completed.")
summary_lines.append("")
summary_lines.append("Exact paper-style features used:")
summary_lines.append("1. Variants grouped by gene.")
summary_lines.append("2. ALT allele treated as mutation allele.")
summary_lines.append("3. Variant frequency estimated from unaffected/low phenotype samples.")
summary_lines.append("4. Haploid-smoothed q = (m_U + 1) / (n_U + 2).")
summary_lines.append("5. Per-sample genetic score = sum ALT_ij / sqrt(n_total*q_j*(1-q_j)).")
summary_lines.append("6. Samples ranked by genetic score.")
summary_lines.append("7. Rank sum computed among affected/high phenotype samples.")
summary_lines.append("8. Affected/unaffected labels permuted; weights and scores recomputed each permutation.")
summary_lines.append("")
summary_lines.append(f"PC50 affected/high threshold: PC50 >= {PC50_THRESHOLD}")
summary_lines.append(f"RSA affected/high threshold: RSA >= {RSA_THRESHOLD}")
summary_lines.append("")
summary_lines.append(f"PC50 genes tested: {pc50_res.shape[0]}")
summary_lines.append(f"RSA genes tested: {rsa_res.shape[0]}")
summary_lines.append("")
summary_lines.append("Top PC50 hits:")
summary_lines.append(
    pc50_res.head(10)[[
        "Gene_ID", "Common_name", "CHROM", "N_SNPs_used",
        "MB_z", "MB_p_perm_one_sided", "MB_FDR_BH"
    ]].to_string(index=False)
)
summary_lines.append("")
summary_lines.append("Top RSA hits:")
summary_lines.append(
    rsa_res.head(10)[[
        "Gene_ID", "Common_name", "CHROM", "N_SNPs_used",
        "MB_z", "MB_p_perm_one_sided", "MB_FDR_BH"
    ]].to_string(index=False)
)
summary_lines.append("")
summary_lines.append("KIC1 detail:")
summary_lines.append(kic_summary_df.to_string(index=False))

with open(OUTDIR / "analysis_summary_Madsen_Browning.txt", "w") as f:
    f.write("\n".join(summary_lines))

run_meta = {
    "genome_csv": str(GENOME_CSV),
    "kic_csv": str(KIC_CSV),
    "method": "Madsen-Browning 2009 weighted-sum rank-sum permutation test",
    "haploid_adaptation": "ALT present/absent; q=(m_U+1)/(n_U+2)",
    "pc50_threshold_affected": PC50_THRESHOLD,
    "rsa_threshold_affected": RSA_THRESHOLD,
    "genome_n_perm": GENOME_N_PERM,
    "kic1_n_perm": KIC1_N_PERM,
    "min_alt_count": MIN_ALT_COUNT,
    "min_ref_count": MIN_REF_COUNT,
    "min_snps_per_gene": MIN_SNPS_PER_GENE,
    "max_intergenic_distance_bp": MAX_DIST,
    "kic1_gene": KIC1_GENE,
    "pc50_genes_tested": int(pc50_res.shape[0]),
    "rsa_genes_tested": int(rsa_res.shape[0]),
}

with open(OUTDIR / "run_metadata_Madsen_Browning.json", "w") as f:
    json.dump(run_meta, f, indent=2)

print("\nDONE.")
print("Outputs saved in:")
print(OUTDIR)

print("\nMain output files:")
print(" - PC50_genomewide_Madsen_Browning_results.csv")
print(" - RSA_genomewide_Madsen_Browning_results.csv")
print(" - KIC1_Madsen_Browning_detail_summary.csv")
print(" - KIC1_PC50_Madsen_Browning_sample_scores.csv")
print(" - KIC1_RSA_Madsen_Browning_sample_scores.csv")
print(" - KIC1_PC50_Madsen_Browning_variant_weights.csv")
print(" - KIC1_RSA_Madsen_Browning_variant_weights.csv")
print(" - analysis_summary_Madsen_Browning.txt")
print(" - figures/")

# ============================================================
# ADD VOLCANO PLOTS FOR MADSEN-BROWNING RESULTS
# Paste this AFTER pc50_res and rsa_res are created
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Make sure output folders exist
OUTDIR = Path("KIC1_Madsen_Browning_outputs")
FIGDIR = OUTDIR / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

def save_and_show(path_base):
    """
    Save PNG/PDF and also show inline in notebook.
    """
    png_path = str(path_base) + ".png"
    pdf_path = str(path_base) + ".pdf"

    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.savefig(pdf_path, bbox_inches="tight")

    print("Saved:", png_path)
    print("Saved:", pdf_path)

    plt.show()


def clean_gene_label(row):
    """
    Prefer common gene name if available, otherwise Gene_ID.
    """
    common = str(row.get("Common_name", "")).split(";")[0]

    if common in ["", "nan", "None", "NaN"]:
        return str(row["Gene_ID"])
    else:
        return common


def make_mb_volcano(
    res,
    phenotype_col,
    x_col="MB_z",
    p_col="MB_p_perm_one_sided",
    fdr_col="MB_FDR_BH",
    label_top=15,
    fdr_cutoff=0.10,
    p_cutoff=0.05,
    label_kic1=True
):
    """
    Volcano plot for formal Madsen-Browning paper-style result.

    x-axis:
        MB_z = standardized Madsen-Browning rank-sum statistic

    y-axis:
        -log10(Madsen-Browning permutation p-value)

    Positive MB_z means:
        affected/high phenotype group tends to have higher weighted ALT burden.
    """

    df = res.copy()

    df[x_col] = pd.to_numeric(df[x_col], errors="coerce")
    df[p_col] = pd.to_numeric(df[p_col], errors="coerce")

    if fdr_col in df.columns:
        df[fdr_col] = pd.to_numeric(df[fdr_col], errors="coerce")
    else:
        df[fdr_col] = np.nan

    df = df.dropna(subset=[x_col, p_col]).copy()
    df = df[(df[p_col] > 0) & (df[p_col] <= 1)].copy()

    if df.empty:
        print(f"No valid rows for {phenotype_col} volcano plot.")
        return

    df["minus_log10_p"] = -np.log10(df[p_col])

    # Significance categories
    df["Volcano_group"] = "Not significant"
    df.loc[df[p_col] < p_cutoff, "Volcano_group"] = f"p < {p_cutoff}"

    if fdr_col in df.columns:
        df.loc[df[fdr_col] < fdr_cutoff, "Volcano_group"] = f"FDR < {fdr_cutoff}"

    # Bonferroni line
    bonf_p = 0.05 / df.shape[0]
    bonf_y = -np.log10(bonf_p)

    plt.figure(figsize=(9, 7))

    # Plot groups separately
    for group in ["Not significant", f"p < {p_cutoff}", f"FDR < {fdr_cutoff}"]:
        sub = df[df["Volcano_group"] == group]

        if sub.empty:
            continue

        plt.scatter(
            sub[x_col],
            sub["minus_log10_p"],
            s=45,
            alpha=0.8,
            label=group
        )

    # Reference lines
    plt.axhline(-np.log10(p_cutoff), linestyle="--", linewidth=1.2)
    plt.axhline(bonf_y, linestyle=":", linewidth=1.5)
    plt.axvline(0, linestyle="-", linewidth=0.8)

    # Label top genes by p-value
    top = df.nsmallest(label_top, p_col).copy()

    # Also label KIC1 if present
    if label_kic1 and "Gene_ID" in df.columns:
        kic = df[df["Gene_ID"] == "PF3D7_0606000"]
        if not kic.empty:
            top = pd.concat([top, kic], ignore_index=True).drop_duplicates("Gene_ID")

    for _, row in top.iterrows():
        lab = clean_gene_label(row)

        plt.text(
            row[x_col],
            row["minus_log10_p"] + 0.05,
            lab,
            fontsize=8,
            rotation=30,
            ha="left",
            va="bottom"
        )

    plt.xlabel("Madsen-Browning rank-sum Z statistic")
    plt.ylabel("-log10(Madsen-Browning permutation p)")
    plt.title(
        f"{phenotype_col} Madsen-Browning volcano plot\n"
        "Positive Z = higher weighted ALT burden in affected/high group"
    )

    plt.legend(frameon=False, fontsize=9)
    plt.tight_layout()

    save_and_show(FIGDIR / f"{phenotype_col}_Madsen_Browning_volcano_MB_z")


def make_continuous_burden_volcano(
    res,
    phenotype_col,
    x_col="Continuous_score_slope",
    p_col="Continuous_score_p",
    label_top=15,
    p_cutoff=0.05,
    label_kic1=True
):
    """
    Volcano plot for descriptive continuous burden association.

    x-axis:
        slope from continuous phenotype ~ Madsen-Browning weighted score

    y-axis:
        -log10(OLS p-value)

    This is NOT the original Madsen-Browning paper statistic.
    It is useful because your PC50/RSA phenotypes are continuous.
    """

    df = res.copy()

    if x_col not in df.columns or p_col not in df.columns:
        print(f"Skipping continuous volcano for {phenotype_col}: missing {x_col} or {p_col}")
        return

    df[x_col] = pd.to_numeric(df[x_col], errors="coerce")
    df[p_col] = pd.to_numeric(df[p_col], errors="coerce")

    df = df.dropna(subset=[x_col, p_col]).copy()
    df = df[(df[p_col] > 0) & (df[p_col] <= 1)].copy()

    if df.empty:
        print(f"No valid rows for {phenotype_col} continuous volcano plot.")
        return

    df["minus_log10_p"] = -np.log10(df[p_col])

    bonf_p = 0.05 / df.shape[0]
    bonf_y = -np.log10(bonf_p)

    df["Volcano_group"] = "Not significant"
    df.loc[df[p_col] < p_cutoff, "Volcano_group"] = f"p < {p_cutoff}"
    df.loc[df[p_col] < bonf_p, "Volcano_group"] = "Bonferroni significant"

    plt.figure(figsize=(9, 7))

    for group in ["Not significant", f"p < {p_cutoff}", "Bonferroni significant"]:
        sub = df[df["Volcano_group"] == group]

        if sub.empty:
            continue

        plt.scatter(
            sub[x_col],
            sub["minus_log10_p"],
            s=45,
            alpha=0.8,
            label=group
        )

    plt.axhline(-np.log10(p_cutoff), linestyle="--", linewidth=1.2)
    plt.axhline(bonf_y, linestyle=":", linewidth=1.5)
    plt.axvline(0, linestyle="-", linewidth=0.8)

    top = df.nsmallest(label_top, p_col).copy()

    if label_kic1 and "Gene_ID" in df.columns:
        kic = df[df["Gene_ID"] == "PF3D7_0606000"]
        if not kic.empty:
            top = pd.concat([top, kic], ignore_index=True).drop_duplicates("Gene_ID")

    for _, row in top.iterrows():
        lab = clean_gene_label(row)

        plt.text(
            row[x_col],
            row["minus_log10_p"] + 0.05,
            lab,
            fontsize=8,
            rotation=30,
            ha="left",
            va="bottom"
        )

    plt.xlabel(f"Slope from {phenotype_col} ~ weighted ALT burden score")
    plt.ylabel("-log10(OLS p)")
    plt.title(
        f"{phenotype_col} continuous burden-score volcano plot\n"
        "Descriptive follow-up, not original Madsen-Browning statistic"
    )

    plt.legend(frameon=False, fontsize=9)
    plt.tight_layout()

    save_and_show(FIGDIR / f"{phenotype_col}_continuous_burden_score_volcano")


# ============================================================
# Generate volcano plots
# ============================================================

# Formal Madsen-Browning volcano plots
make_mb_volcano(
    pc50_res,
    phenotype_col="PC50",
    label_top=15,
    fdr_cutoff=0.10,
    p_cutoff=0.05
)

make_mb_volcano(
    rsa_res,
    phenotype_col="RSA",
    label_top=15,
    fdr_cutoff=0.10,
    p_cutoff=0.05
)

# Continuous phenotype descriptive volcano plots
make_continuous_burden_volcano(
    pc50_res,
    phenotype_col="PC50",
    label_top=15,
    p_cutoff=0.05
)

make_continuous_burden_volcano(
    rsa_res,
    phenotype_col="RSA",
    label_top=15,
    p_cutoff=0.05
)

print("\nVolcano plots added.")
print("Saved in:", FIGDIR.resolve())



In [ ]:
# ============================================================
# RANKED KIC1 SCORE PLOT COLORED BY PC50 GROUP
# ============================================================

def plot_ranked_scores_pc50_group(score_df, pc50_df, phenotype_name_for_title, outname, pc50_threshold=5.0):
    plot_df = score_df.copy()
    plot_df["Sample"] = plot_df["Sample"].astype(str)

    pc50_tmp = pc50_df.copy().reset_index()[["Sample", "PC50"]]
    plot_df = plot_df.merge(pc50_tmp, on="Sample", how="left")

    plot_df["PC50_group"] = np.where(
        plot_df["PC50"] >= pc50_threshold,
        f"PC50 ≥ {pc50_threshold}",
        f"PC50 < {pc50_threshold}"
    )

    plot_df = plot_df.sort_values("MB_rank").reset_index(drop=True)

    color_map = {
        f"PC50 < {pc50_threshold}": "purple",
        f"PC50 ≥ {pc50_threshold}": "gold"
    }

    plt.figure(figsize=(12, 5))
    x = np.arange(len(plot_df))

    for grp in [f"PC50 < {pc50_threshold}", f"PC50 ≥ {pc50_threshold}"]:
        sub = plot_df[plot_df["PC50_group"] == grp]
        idx = sub.index.values

        plt.scatter(
            idx,
            sub["MB_weighted_score"],
            color=color_map[grp],
            s=120,
            edgecolor="black",
            linewidth=0.5,
            label=grp
        )

    for xi, (_, row) in zip(x, plot_df.iterrows()):
        plt.text(
            xi,
            row["MB_weighted_score"],
            row["Sample"],
            fontsize=8,
            rotation=45,
            ha="left",
            va="bottom"
        )

    plt.xlabel("Samples ranked by Madsen-Browning genetic score")
    plt.ylabel("KIC1 weighted ALT score")
    plt.title(f"KIC1 {phenotype_name_for_title}: ranked genetic scores\nColored by PC50 group")
    plt.legend(frameon=False)
    plt.tight_layout()

    plt.savefig(str(FIGDIR / f"{outname}.png"), dpi=300, bbox_inches="tight")
    plt.savefig(str(FIGDIR / f"{outname}.pdf"), bbox_inches="tight")
    plt.show()


# Example:
plot_ranked_scores_pc50_group(
    kic_rsa_scores,
    pc50,
    phenotype_name_for_title="RSA",
    outname="KIC1_RSA_ranked_scores_colored_by_PC50_group",
    pc50_threshold=5.0
)